In [ ]:
"""MyoClass 1.0.1: automatic/manual marker calls, inferred category and morphometry.
Adapted from MyoClass_cp3_260126.ipynb and Corrections_napari_v2.py.
Launch in the existing Cellpose 3 environment:
    conda install -c conda-forge voila ipywidgets ipyfilechooser openpyxl napari pyqt
    voila MyoClass_v1.0.1.ipynb
Napari opens a desktop window on the computer running Voilà. Cellpose inference
and real-tissue classification require validation with your trained model.
"""
import asyncio
import hashlib
import html
import io
import json
import math
import os
import re
import shutil
import subprocess
import sys
import threading
import traceback
import uuid
import xml.etree.ElementTree as ET
from datetime import datetime, timezone
from importlib.metadata import version, PackageNotFoundError
from pathlib import Path
import numpy as np
import pandas as pd
import tifffile as tiff
from scipy.ndimage import binary_erosion, gaussian_filter
from scipy.spatial import ConvexHull, QhullError
from skimage.measure import regionprops
from skimage.segmentation import find_boundaries
from skimage.transform import resize
from PIL import Image

APP_VERSION = "1.0.1"


def installed_version(package):
    try:
        return version(package)
    except PackageNotFoundError:
        return "not installed"


def utc_now():
    return datetime.now(timezone.utc).isoformat(timespec="seconds")


def signature(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False,
                                     allow_nan=False).encode("utf-8")).hexdigest()


def file_stamp(path):
    path = Path(path).resolve()
    info = path.stat()
    return {"path": str(path), "size": info.st_size, "mtime_ns": info.st_mtime_ns}


def inside(path, parent):
    try:
        Path(path).resolve().relative_to(Path(parent).resolve())
        return True
    except ValueError:
        return False


def atomic_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_name(f".{path.stem}.{uuid.uuid4().hex}.tmp")
    try:
        temp.write_text(json.dumps(value, ensure_ascii=False, indent=2,
                                   allow_nan=False), encoding="utf-8")
        os.replace(temp, path)
    finally:
        temp.unlink(missing_ok=True)


def atomic_csv(path, frame):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_name(f".{path.stem}.{uuid.uuid4().hex}.csv")
    try:
        frame.to_csv(temp, index=False, encoding="utf-8-sig")
        os.replace(temp, path)
    finally:
        temp.unlink(missing_ok=True)


def write_image(path, array, spacing):
    """Write a single-plane TIFF with explicit physical calibration in µm."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_name(f".{path.stem}.{uuid.uuid4().hex}.tif")
    try:
        tiff.imwrite(temp, array, ome=True, photometric="minisblack",
                     bigtiff=array.nbytes >= 2**32 - 2**25,
                     metadata={"axes": "YX", "PhysicalSizeY": spacing[0],
                               "PhysicalSizeX": spacing[1],
                               "PhysicalSizeYUnit": "µm", "PhysicalSizeXUnit": "µm"})
        os.replace(temp, path)
    finally:
        temp.unlink(missing_ok=True)


def ome_spacing(xml, scene):
    if not xml:
        return None
    try:
        images = [node for node in ET.fromstring(xml) if node.tag.endswith("}Image")]
        pixels = next(node for node in images[scene] if node.tag.endswith("}Pixels"))
        multipliers = {"µm": 1., "μm": 1., "um": 1., "nm": .001,
                       "mm": 1000., "m": 1e6}
        result = []
        for axis in ("Y", "X"):
            unit = pixels.attrib.get(f"PhysicalSize{axis}Unit", "µm")
            if unit not in multipliers:
                return None
            result.append(float(pixels.attrib[f"PhysicalSize{axis}"]) * multipliers[unit])
        return tuple(result) if all(np.isfinite(result)) and min(result) > 0 else None
    except (ET.ParseError, KeyError, IndexError, StopIteration, ValueError):
        return None


def select_tiff_plane(array, axes, settings):
    """Use named axes, with an explicit override for unannotated 3D TIFFs."""
    axes = axes.upper()
    if settings["axes"] != "auto" and array.ndim != 2:
        if array.ndim != 3:
            raise ValueError("The manual TIFF layout applies to 3D arrays only. Use Auto for OME multidimensional files.")
        axes = settings["axes"]
    if len(axes) != array.ndim or len(set(axes)) != len(axes):
        raise ValueError(f"Unsupported TIFF axes {axes} for shape {array.shape}.")
    # RGB samples are a channel axis; do not confuse them with an OME scene.
    if "S" in axes and "C" not in axes:
        axes = axes.replace("S", "C")
    channel_count = array.shape[axes.index("C")] if "C" in axes else 1
    selected_channel = settings["channel"] if channel_count > 1 else 0
    selectors, remaining = [], []
    for axis, size in zip(axes, array.shape):
        if axis in "YX":
            selectors.append(slice(None))
            remaining.append(axis)
            continue
        index = {"C": selected_channel, "T": settings["time"], "Z": settings["z"]}.get(axis)
        if index is None:
            if size != 1:
                raise ValueError(f"Ambiguous TIFF axis '{axis}' in shape {array.shape}. Select Channels first or Channels last if this is a channel stack.")
            index = 0
        if not 0 <= index < size:
            raise ValueError(f"Axis {axis}: requested index {index}, but only {size} position(s) exist (numbering starts at 0).")
        selectors.append(index)
    plane = array[tuple(selectors)]
    if remaining == ["X", "Y"]:
        plane = plane.T
    elif remaining != ["Y", "X"]:
        raise ValueError(f"Expected Y/X spatial axes, found {remaining}.")
    return plane, {"axes": axes, "channels": channel_count, "channel_used": selected_channel}


def load_plane(path, scene, settings):
    metadata_spacing = None
    if path.suffix.lower() == ".czi":
        try:
            from aicsimageio import AICSImage
        except ImportError as exc:
            raise RuntimeError("CZI input requires aicsimageio and aicspylibczi in this Python environment.") from exc
        image = AICSImage(str(path))
        image.set_scene(scene)
        selected = settings["channel"] if image.dims.C > 1 else 0
        for axis, index in (("C", selected), ("T", settings["time"]), ("Z", settings["z"])):
            size = getattr(image.dims, axis)
            if not 0 <= index < size:
                raise ValueError(f"Axis {axis}: requested index {index}, but only {size} position(s) exist.")
        plane = image.get_image_dask_data("YX", C=selected, T=settings["time"], Z=settings["z"]).compute()
        physical = image.physical_pixel_sizes
        if physical.Y and physical.X:
            metadata_spacing = (float(physical.Y), float(physical.X))
        info = {"axes": image.dims.order, "channels": image.dims.C, "channel_used": selected}
    else:
        with tiff.TiffFile(path) as image:
            series = image.series[scene]
            plane, info = select_tiff_plane(series.asarray(), series.axes, settings)
            metadata_spacing = ome_spacing(image.ome_metadata, scene)
    plane = np.asarray(plane)
    if plane.ndim != 2 or min(plane.shape) < 2:
        raise ValueError(f"Expected a two-dimensional plane, found {plane.shape}.")
    if settings["calibration"] == "metadata" and metadata_spacing is not None:
        spacing, calibration_source = metadata_spacing, "Image metadata"
    else:
        spacing = (settings["pixel_size"], settings["pixel_size"])
        calibration_source = "Manual" if settings["calibration"] == "manual" else "Manual fallback (metadata unavailable)"
    info.update({"original_shape": list(plane.shape), "spacing_yx_um": list(spacing),
                 "calibration_source": calibration_source})
    return plane, info


def normalize_nonzero_2d(array):
    array = np.nan_to_num(np.asarray(array, dtype=np.float32), nan=0., posinf=0., neginf=0.)
    array = np.maximum(array, 0.)
    result = np.zeros_like(array)
    valid = array > 0
    if valid.any():
        low, high = float(array[valid].min()), float(array[valid].max())
        result[valid] = (array[valid] - low) / (high - low) if high > low else 1.
    return result


def validate_mask(mask):
    mask = np.asarray(mask)
    if mask.ndim != 2 or mask.dtype.kind not in "bui":
        raise ValueError(f"Label masks must be 2D integer TIFFs; found shape={mask.shape}, dtype={mask.dtype}.")
    if mask.size and (mask.min() < 0 or mask.max() > np.iinfo(np.uint32).max):
        raise ValueError("Labels must be nonnegative 32-bit integers (0 is background).")
    return mask.astype(np.uint32, copy=False)


def minimum_feret(region, spacing):
    """Minimum hull width of pixel centres, as in the source notebook.

    Using boundary pixel centres gives the same hull as all region pixels.
    Degenerate (point/line) regions have a minimum width of zero.
    """
    boundary = region.image & ~binary_erosion(region.image, border_value=0)
    points = np.argwhere(boundary).astype(float) * np.asarray(spacing)
    if len(points) < 3:
        return 0.
    try:
        hull = ConvexHull(points)
    except QhullError:
        return 0.
    vertices = points[hull.vertices]
    # Translation improves numerical conditioning without changing widths.
    vertices -= vertices.mean(axis=0)
    edges = np.roll(vertices, -1, axis=0) - vertices
    lengths = np.linalg.norm(edges, axis=1)
    normals = np.column_stack((-edges[:, 1], edges[:, 0])) / lengths[:, None]
    widths = [np.ptp(vertices @ normal) for normal in normals]
    return float(min(widths))


def distribution(values, maximum, step, unit):
    edges = np.linspace(0, maximum, int(round(maximum / step)) + 1)
    names = [f"{a:g}-{b:g} {unit}" for a, b in zip(edges[:-1], edges[1:])]
    names.append(f">={maximum:g} {unit}")
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    indices = np.searchsorted(edges, values, side="right") - 1
    counts = np.bincount(indices, minlength=len(names)) if len(values) else np.zeros(len(names), dtype=int)
    count_map = {key: int(value) for key, value in zip(names, counts)}
    pct_map = {key: float(value / len(values) * 100) if len(values) else 0.
               for key, value in zip(names, counts)}
    return count_map, pct_map


def preview_png(plane, mask=None, max_size=1400):
    """A bounded preview avoids embedding full-resolution tissue images."""
    height, width = plane.shape
    scale = min(1., max_size / max(height, width))
    shape = (max(1, int(round(height * scale))), max(1, int(round(width * scale))))
    # Reduce before making temporary floating-point/RGB arrays for large tissue mosaics.
    factor = max(1, max(height, width) // max_size)
    sampled = np.asarray(plane[::factor, ::factor], dtype=np.float32)
    base = resize(sampled, shape, order=1, preserve_range=True,
                  anti_aliasing=True) if shape != sampled.shape else sampled
    finite = base[np.isfinite(base)]
    lo, hi = np.percentile(finite, (1, 99)) if finite.size else (0., 1.)
    grey = np.clip((np.nan_to_num(base) - lo) / (hi - lo), 0, 1) if hi > lo else np.zeros(shape)
    rgb = np.repeat((grey * 255).astype(np.uint8)[..., None], 3, axis=2)
    if mask is not None:
        # Preview boundaries are computed on reduced labels; measurements use the full mask.
        small_mask = mask[::factor, ::factor]
        if shape != small_mask.shape:
            small_mask = resize(small_mask, shape, order=0, preserve_range=True,
                                anti_aliasing=False).astype(np.uint32)
        boundaries = find_boundaries(small_mask, mode="inner")
        rgb[boundaries] = (255, 90, 110)
    stream = io.BytesIO()
    Image.fromarray(rgb).save(stream, format="PNG")
    return stream.getvalue()



In [ ]:
REVIEW_SCRIPT_SOURCE = '"""MyoClass manual review. Launch from Voilà or with --session review_session.json.\n\nBinary marker calls are edited; reference labels and centroid positions stay fixed.\nQt and napari are imported only when opening the desktop editor.\n"""\nimport argparse\nimport hashlib\nimport json\nimport os\nimport re\nimport shutil\nimport uuid\nfrom datetime import datetime, timezone\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nimport tifffile as tiff\n\n\ndef channel_mode(spec):\n    """Read explicit modes and the legacy derived flag without altering old projects."""\n    return spec.get("mode", "inferred" if spec.get("derived") else "automatic")\n\n\ndef normalize_channel_specs(channels, infer_last=False):\n    specs = [dict(spec) for spec in channels]\n    for i, spec in enumerate(specs):\n        legacy_inferred = infer_last and i == len(specs) - 1\n        spec["mode"] = spec.get("mode", "inferred" if spec.get("derived") or legacy_inferred else "automatic")\n        if spec["mode"] not in ("automatic", "manual", "inferred"):\n            raise ValueError("Marker mode must be Automatic, Manual or Inferred.")\n        spec["derived"] = spec["mode"] == "inferred"\n    inferred = [spec for spec in specs if spec["derived"]]\n    if len(inferred) > 1:\n        raise ValueError("Only one category can be inferred from the other enabled markers.")\n    if inferred and len(specs) < 2:\n        raise ValueError("An inferred category needs at least one automatic or manual marker.")\n    return specs\n\n\ndef recompute_inferred(frame, channels):\n    """Inferred positivity is the absence of every automatic/manual marker."""\n    sources = [f"ch{spec[\'index\']}" for spec in channels if channel_mode(spec) != "inferred"]\n    for spec in channels:\n        if channel_mode(spec) == "inferred":\n            if not sources:\n                raise ValueError("An inferred category needs a source marker.")\n            frame[f"ch{spec[\'index\']}"] = (frame[sources].sum(axis=1) == 0).astype(np.uint8)\n    return frame\n\n\ndef digest_file(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for block in iter(lambda: stream.read(2**20), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef save_json(path, payload):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temp = path.with_name(f".{path.name}.{uuid.uuid4().hex}.tmp")\n    try:\n        temp.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")\n        os.replace(temp, path)\n    finally:\n        temp.unlink(missing_ok=True)\n\n\ndef save_mapping(path, frame):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temp = path.with_name(f".{path.stem}.{uuid.uuid4().hex}.xlsx")\n    try:\n        frame.to_excel(temp, index=False, sheet_name="mapping", engine="openpyxl")\n        os.replace(temp, path)\n    finally:\n        temp.unlink(missing_ok=True)\n\n\ndef load_mapping(path, entry, automatic=None):\n    frame = pd.read_excel(path, sheet_name="mapping", engine="openpyxl")\n    channels = [f"ch{c[\'index\']}" for c in entry["channels"]]\n    required = ["label_id", "area_px", "y", "x", *channels]\n    if not set(required).issubset(frame.columns):\n        raise ValueError(f"Mapping is missing columns: {set(required) - set(frame.columns)}")\n    ids = pd.to_numeric(frame["label_id"], errors="raise").to_numpy(dtype=float)\n    if not np.isfinite(ids).all() or not np.equal(ids, np.floor(ids)).all() or (ids <= 0).any():\n        raise ValueError("label_id must contain positive integer identifiers.")\n    frame["label_id"] = ids.astype(np.int64)\n    if frame.label_id.duplicated().any():\n        raise ValueError("Duplicate label_id values in mapping.")\n    frame = frame.sort_values("label_id").reset_index(drop=True)\n    for channel in channels:\n        values = pd.to_numeric(frame[channel], errors="raise")\n        if not values.isin([0, 1]).all():\n            raise ValueError(f"{channel} must contain only 0 and 1, with no missing values.")\n        frame[channel] = values.astype(np.uint8)\n    if not np.isfinite(frame[["area_px", "x", "y"]].to_numpy(dtype=float)).all():\n        raise ValueError("Mapping coordinates and areas must be finite.")\n    if automatic is not None:\n        if not np.array_equal(frame.label_id, automatic.label_id):\n            raise ValueError("Review label IDs differ from the automatic mapping.")\n        if not np.allclose(frame[["area_px", "y", "x"]], automatic[["area_px", "y", "x"]], rtol=0, atol=1e-6):\n            raise ValueError("Areas or centroid coordinates changed. Only marker calls may be edited.")\n    return frame\n\n\ndef get_review_state(entry):\n    path = Path(entry["review_path"])\n    meta_path = Path(entry["review_meta"])\n    if not path.is_file() or not meta_path.is_file():\n        return "Missing review"\n    try:\n        meta = json.loads(meta_path.read_text(encoding="utf-8"))\n        if meta.get("automatic_signature") != entry["map_sig"]:\n            return "Stale review"\n        if meta.get("mapping_hash") != digest_file(path):\n            return "Modified outside editor"\n        return "Reviewed" if meta.get("reviewed") else "Draft"\n    except (OSError, ValueError):\n        return "Invalid review metadata"\n\n\ndef initialize_review(entry, reset=False):\n    path = Path(entry["review_path"])\n    if path.exists() and not reset:\n        return\n    frame = load_mapping(entry["automatic_path"], entry)\n    if path.exists():\n        backup = path.parent / "Backups" / f"{path.stem}_{datetime.now().strftime(\'%Y%m%d_%H%M%S_%f\')}.xlsx"\n        backup.parent.mkdir(parents=True, exist_ok=True)\n        shutil.copy2(path, backup)\n        meta_path = Path(entry["review_meta"])\n        if meta_path.exists():\n            shutil.copy2(meta_path, backup.with_suffix(".json"))\n    save_mapping(path, frame)\n    save_json(entry["review_meta"], {\n        "automatic_signature": entry["map_sig"], "mapping_hash": digest_file(path),\n        "reviewed": False, "saved_utc": datetime.now(timezone.utc).isoformat(),\n        "image_id": entry["image_id"]})\n\n\nclass ReviewDocument:\n    """File-backed state independent of napari, so corrections can be validated."""\n    def __init__(self, entry):\n        self.entry = entry\n        state = get_review_state(entry)\n        if state in ("Stale review", "Missing review", "Invalid review metadata"):\n            raise ValueError(f"{state}. Use Reset selected review in MyoClass to create a fresh draft (backs up existing edits).")\n        if digest_file(entry["automatic_path"]) != entry["automatic_hash"]:\n            raise ValueError("The automatic mapping was modified. Run automatic classification again.")\n        self.automatic = load_mapping(entry["automatic_path"], entry)\n        self.frame = load_mapping(entry["review_path"], entry, self.automatic)\n        self.expected_hash = digest_file(entry["review_path"])\n        self.channels = [f"ch{c[\'index\']}" for c in entry["channels"]]\n        self.dirty = False\n        self.was_reviewed = state == "Reviewed"\n        self.refresh_inferred()\n\n    def refresh_inferred(self):\n        before = self.frame[self.channels].copy()\n        recompute_inferred(self.frame, self.entry["channels"])\n        if not before.equals(self.frame[self.channels]):\n            self.dirty = True\n\n    @property\n    def points(self):\n        return self.frame[["y", "x"]].to_numpy(dtype=float)\n\n    def set_types(self, channel, indices, value):\n        if channel not in self.channels or value not in (0, 1):\n            raise ValueError("Select a valid marker and a binary class (0 or 1).")\n        spec = next(c for c in self.entry["channels"] if f"ch{c[\'index\']}" == channel)\n        if channel_mode(spec) == "inferred":\n            raise ValueError("This category is inferred. Edit its automatic/manual source markers instead.")\n        indices = np.asarray(sorted(set(indices)), dtype=int)\n        if len(indices) and (indices.min() < 0 or indices.max() >= len(self.frame)):\n            raise ValueError("Invalid point selection.")\n        if len(indices) and (self.frame.loc[indices, channel] != value).any():\n            self.frame.loc[indices, channel] = value\n            self.dirty = True\n            self.refresh_inferred()\n\n    def save(self, reviewed=False):\n        self.refresh_inferred()\n        path = Path(self.entry["review_path"])\n        if digest_file(path) != self.expected_hash:\n            raise RuntimeError("This review file changed in another window. Close this editor and reload it before saving.")\n        if self.dirty:\n            backup = path.parent / "Backups" / f"{path.stem}_{datetime.now().strftime(\'%Y%m%d_%H%M%S_%f\')}.xlsx"\n            backup.parent.mkdir(parents=True, exist_ok=True)\n            shutil.copy2(path, backup)\n        final_reviewed = reviewed or (self.was_reviewed and not self.dirty)\n        save_mapping(path, self.frame)\n        self.expected_hash = digest_file(path)\n        save_json(self.entry["review_meta"], {\n            "automatic_signature": self.entry["map_sig"], "mapping_hash": self.expected_hash,\n            "reviewed": final_reviewed, "saved_utc": datetime.now(timezone.utc).isoformat(),\n            "image_id": self.entry["image_id"], "channels": self.channels})\n        self.dirty = False\n        self.was_reviewed = final_reviewed\n\n\ndef launch_editor(session_path=None, selected_id=None):\n    import napari\n    from qtpy.QtCore import Qt, QObject, QEvent\n    from qtpy.QtWidgets import (QWidget, QVBoxLayout, QHBoxLayout, QLabel, QPushButton,\n                               QComboBox, QCheckBox, QSlider, QFileDialog, QMessageBox)\n\n    viewer = napari.Viewer(title="MyoClass · manual fiber typing")\n    if not session_path:\n        session_path, _ = QFileDialog.getOpenFileName(None, "Select a MyoClass review_session.json", "", "MyoClass session (*.json)")\n        if not session_path:\n            viewer.close()\n            return\n    payload = json.loads(Path(session_path).read_text(encoding="utf-8"))\n    entries = payload["entries"]\n    if not entries:\n        raise ValueError("This review session has no valid classified images.")\n\n    class Editor:\n        def __init__(self):\n            self.doc = None\n            self.points_layer = None\n            self.marker_images = {}\n            self.current_channel = None\n            self.restoring = False\n            panel = QWidget()\n            layout = QVBoxLayout(panel)\n            layout.addWidget(QLabel("MyoClass · fluorescence marker review"))\n            self.image_combo = QComboBox()\n            for entry in entries:\n                self.image_combo.addItem(f"{entry[\'filename\']} · scene {entry[\'scene\'] + 1}", entry["image_id"])\n            layout.addWidget(self.image_combo)\n            nav = QHBoxLayout()\n            previous = QPushButton("Previous")\n            following = QPushButton("Next")\n            previous.clicked.connect(lambda: self.navigate(-1))\n            following.clicked.connect(lambda: self.navigate(1))\n            nav.addWidget(previous); nav.addWidget(following)\n            layout.addLayout(nav)\n            layout.addWidget(QLabel("Marker to view / edit:"))\n            self.channel_combo = QComboBox()\n            layout.addWidget(self.channel_combo)\n            self.show_all = QCheckBox("Show all fluorescence channels")\n            self.show_all.stateChanged.connect(self.apply_image_visibility)\n            layout.addWidget(self.show_all)\n            layout.addWidget(QLabel("Visible points:"))\n            self.filter_combo = QComboBox()\n            self.filter_combo.addItems(["All", "Negative (0)", "Positive (1)"])\n            self.filter_combo.currentIndexChanged.connect(self.refresh_points)\n            layout.addWidget(self.filter_combo)\n            actions = QHBoxLayout()\n            zero, one = QPushButton("F1 · negative (0)"), QPushButton("F2 · positive (1)")\n            self.zero_button, self.one_button = zero, one\n            zero.clicked.connect(lambda: self.set_type(0))\n            one.clicked.connect(lambda: self.set_type(1))\n            actions.addWidget(zero); actions.addWidget(one)\n            layout.addLayout(actions)\n            self.show_ids = QCheckBox("Show fiber label IDs")\n            self.show_ids.stateChanged.connect(self.refresh_points)\n            layout.addWidget(self.show_ids)\n            layout.addWidget(QLabel("Point size"))\n            self.point_size = QSlider(Qt.Horizontal)\n            self.point_size.setRange(3, 150); self.point_size.setValue(25)\n            self.point_size.valueChanged.connect(self.refresh_points)\n            layout.addWidget(self.point_size)\n            layout.addWidget(QLabel("Text size"))\n            self.text_size = QSlider(Qt.Horizontal)\n            self.text_size.setRange(6, 60); self.text_size.setValue(14)\n            self.text_size.valueChanged.connect(self.refresh_points)\n            layout.addWidget(self.text_size)\n            save_row = QHBoxLayout()\n            save = QPushButton("Save draft (Ctrl+S)")\n            done = QPushButton("Mark image reviewed")\n            save.clicked.connect(lambda: self.save(False))\n            done.clicked.connect(lambda: self.save(True))\n            save_row.addWidget(save); save_row.addWidget(done)\n            layout.addLayout(save_row)\n            self.status = QLabel()\n            self.status.setWordWrap(True)\n            layout.addWidget(self.status)\n            help_label = QLabel("Select centroids, then press F1 or F2.\\nManual markers start negative: check every fiber.\\nAutomatic/manual markers are independent; hybrids are allowed.\\nThe inferred category updates after each edit and stays read-only.\\nCentroid positions and label masks stay fixed.\\nMark reviewed only after checking all source markers.")\n            help_label.setWordWrap(True)\n            layout.addWidget(help_label)\n            viewer.window.add_dock_widget(panel, name="MyoClass review", area="left")\n            self.image_combo.currentIndexChanged.connect(self.load_image)\n            self.channel_combo.currentIndexChanged.connect(self.change_channel)\n            index = next((i for i, entry in enumerate(entries) if entry["image_id"] == selected_id), 0)\n            self.image_combo.blockSignals(True); self.image_combo.setCurrentIndex(index); self.image_combo.blockSignals(False)\n            self.load_image(index)\n            for owner in (viewer,):\n                owner.bind_key("F1", lambda _: self.set_type(0), overwrite=True)\n                owner.bind_key("F2", lambda _: self.set_type(1), overwrite=True)\n                owner.bind_key("Control-S", lambda _: self.save(False), overwrite=True)\n                owner.bind_key("s", lambda _: self.save(False), overwrite=True)\n\n        def save(self, reviewed=False):\n            if self.doc is None:\n                return False\n            try:\n                self.doc.save(reviewed)\n                self.update_status()\n                return True\n            except Exception as exc:\n                QMessageBox.critical(None, "Cannot save review", str(exc))\n                return False\n\n        def update_status(self):\n            if self.doc:\n                state = "Unsaved changes" if self.doc.dirty else get_review_state(self.doc.entry)\n                marker = self.current_channel or ""\n                self.status.setText(f"{state} · {len(self.doc.frame)} fibers · {marker}\\n{self.doc.entry[\'review_path\']}")\n\n        def load_image(self, index):\n            if self.doc is not None and self.doc.dirty and not self.save(False):\n                self.image_combo.blockSignals(True)\n                old = next(i for i,e in enumerate(entries) if e[\'image_id\'] == self.doc.entry[\'image_id\'])\n                self.image_combo.setCurrentIndex(old); self.image_combo.blockSignals(False)\n                return\n            try:\n                doc = ReviewDocument(entries[index])\n                mask = tiff.imread(doc.entry["reference_path"])\n                expected = np.unique(mask); expected = expected[expected > 0]\n                if not np.array_equal(expected, doc.frame.label_id.to_numpy()):\n                    raise ValueError("Reference mask label IDs do not match the mapping.")\n                self.doc = doc\n                self.points_layer = None\n                viewer.layers.clear()\n                self.marker_images = {}\n                for spec in doc.entry["channels"]:\n                    channel = f"ch{spec[\'index\']}"\n                    file = doc.entry["prepared_paths"].get(channel)\n                    if file:\n                        plane = tiff.imread(file)\n                        if plane.shape != mask.shape:\n                            raise ValueError(f"{channel}: fluorescence and reference mask shapes differ.")\n                        self.marker_images[channel] = viewer.add_image(plane, name=f"{channel} · {spec[\'name\']}", colormap="gray", visible=False)\n                labels = viewer.add_labels(mask, name="Reference fiber boundaries", opacity=.5)\n                labels.contour = 1\n                labels.editable = False\n                channel = f"ch{doc.entry[\'channels\'][0][\'index\']}"\n                self.current_channel = channel\n                features = {"type": doc.frame[channel].astype(str).to_numpy(), "label_id": doc.frame.label_id.to_numpy()}\n                self.points_layer = viewer.add_points(doc.points, features=features, size=self.point_size.value(),\n                                                      name="Fiber calls · select and press F1/F2",\n                                                      face_color="red", text={"string": "{type}", "color": "white", "size": self.text_size.value()})\n                self.points_layer.mode = "select"\n                self.points_layer.events.data.connect(self.restore_points)\n                for key, callback in [("F1",lambda _:self.set_type(0)),("F2",lambda _:self.set_type(1)),\n                                      ("Control-S",lambda _:self.save(False)),("s",lambda _:self.save(False))]:\n                    self.points_layer.bind_key(key, callback, overwrite=True)\n                self.channel_combo.blockSignals(True)\n                self.channel_combo.clear()\n                for spec in doc.entry["channels"]:\n                    suffix = {"automatic": " (automatic; editable)", "manual": " (manual)", "inferred": " (inferred; read-only)"}[channel_mode(spec)]\n                    self.channel_combo.addItem(f"ch{spec[\'index\']} · {spec[\'name\']}{suffix}", f"ch{spec[\'index\']}")\n                self.channel_combo.setCurrentIndex(0); self.channel_combo.blockSignals(False)\n                self.refresh_points()\n                self.apply_image_visibility()\n                viewer.reset_view()\n            except Exception as exc:\n                QMessageBox.critical(None, "Cannot load image", str(exc))\n\n        def restore_points(self, _=None):\n            if self.restoring or self.doc is None or self.points_layer is None:\n                return\n            if self.points_layer.data.shape != self.doc.points.shape or not np.allclose(self.points_layer.data, self.doc.points):\n                self.restoring = True\n                try:\n                    self.points_layer.data = self.doc.points\n                    self.refresh_points()\n                finally:\n                    self.restoring = False\n\n        def change_channel(self, _=None):\n            if not self.doc or self.channel_combo.currentIndex() < 0:\n                return\n            if self.doc.dirty and not self.save(False):\n                self.channel_combo.blockSignals(True)\n                self.channel_combo.setCurrentIndex(self.doc.channels.index(self.current_channel))\n                self.channel_combo.blockSignals(False)\n                return\n            self.current_channel = self.channel_combo.currentData()\n            self.refresh_points(); self.apply_image_visibility()\n\n        def apply_image_visibility(self, *_):\n            for channel, layer in self.marker_images.items():\n                no_own_image = self.current_channel not in self.marker_images\n                layer.visible = self.show_all.isChecked() or no_own_image or channel == self.current_channel\n\n        def refresh_points(self, *_):\n            if self.points_layer is None or self.doc is None:\n                return\n            spec = next(c for c in self.doc.entry["channels"] if f"ch{c[\'index\']}" == self.current_channel)\n            editable = channel_mode(spec) != "inferred"\n            self.zero_button.setEnabled(editable)\n            self.one_button.setEnabled(editable)\n            values = self.doc.frame[self.current_channel].to_numpy(dtype=int)\n            layer = self.points_layer\n            layer.features = {"type": values.astype(str), "label_id": self.doc.frame.label_id.to_numpy()}\n            colors = np.zeros((len(values),4), dtype=float)\n            colors[:] = (1., .15, .15, .8)\n            colors[values == 1] = (.15, 1., .3, .8)\n            layer.face_color = colors\n            layer.size = self.point_size.value()\n            template = "{label_id}: {type}" if self.show_ids.isChecked() else "{type}"\n            layer.text = {"string": template, "color": "white", "size": self.text_size.value(), "anchor": "center"}\n            choice = self.filter_combo.currentIndex()\n            layer.shown = np.ones(len(values), dtype=bool) if choice == 0 else values == choice-1\n            layer.selected_data = set()\n            layer.refresh()\n            self.update_status()\n\n        def set_type(self, value):\n            if self.doc is None or self.points_layer is None:\n                return\n            spec = next(c for c in self.doc.entry["channels"] if f"ch{c[\'index\']}" == self.current_channel)\n            if channel_mode(spec) == "inferred":\n                self.status.setText("Read-only inferred category: edit the source markers to update these calls.")\n                return\n            indices = set(self.points_layer.selected_data)\n            if indices:\n                self.doc.set_types(self.current_channel, indices, value)\n                self.refresh_points()\n\n        def navigate(self, delta):\n            index = max(0,min(len(entries)-1,self.image_combo.currentIndex()+delta))\n            self.image_combo.setCurrentIndex(index)\n\n    editor = Editor()\n    class CloseGuard(QObject):\n        def eventFilter(self, watched, event):\n            if event.type() == QEvent.Close and editor.doc is not None and editor.doc.dirty:\n                answer = QMessageBox.question(None,"Unsaved marker calls","Save the current draft before closing?",\n                                              QMessageBox.Save | QMessageBox.Discard | QMessageBox.Cancel, QMessageBox.Save)\n                if answer == QMessageBox.Cancel or (answer == QMessageBox.Save and not editor.save(False)):\n                    event.ignore()\n                    return True\n            return False\n    close_guard = CloseGuard(viewer.window._qt_window)\n    viewer.window._qt_window.installEventFilter(close_guard)\n    # Keep editor objects alive throughout the desktop event loop.\n    viewer.window._qt_window._myoclass_editor = editor\n    viewer.window._qt_window._myoclass_close_guard = close_guard\n    napari.run()\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--session", help="MyoClass review_session.json produced by the notebook")\n    parser.add_argument("--image-id", help="Image to select when opening the batch")\n    parser.add_argument("--validate-session", action="store_true", help="Validate review tables without opening napari")\n    args = parser.parse_args()\n    if args.validate_session:\n        if not args.session:\n            parser.error("--validate-session requires --session")\n        payload=json.loads(Path(args.session).read_text(encoding="utf-8"))\n        for entry in payload["entries"]:\n            ReviewDocument(entry)\n        print(f"Validated {len(payload[\'entries\'])} review table(s).")\n    else:\n        launch_editor(args.session, args.image_id)\n'
review_namespace={"__name__":"myoclass_review_embedded"}
exec(compile(REVIEW_SCRIPT_SOURCE,"Corrections_napari_MyoClass.py","exec"),review_namespace)
ReviewDocument=review_namespace["ReviewDocument"]
digest_file=review_namespace["digest_file"]
load_mapping=review_namespace["load_mapping"]
save_mapping=review_namespace["save_mapping"]
get_review_state=review_namespace["get_review_state"]
initialize_review=review_namespace["initialize_review"]

channel_mode=review_namespace["channel_mode"]

normalize_channel_specs=review_namespace["normalize_channel_specs"]

recompute_inferred=review_namespace["recompute_inferred"]


In [ ]:
"""MyoClass workflow: channel-wise Cellpose signal masks -> fiber overlap calls.

Reference fiber IDs remain unchanged throughout automatic typing and review.
Marker calls are multilabel, not a forced choice between mutually exclusive types.
"""


def project_paths(settings):
    raw = settings.get("output_dir", "").strip()
    if not raw:
        if not settings.get("input_dir"):
            raise ValueError("Select an input folder or enter an existing MyoClass output folder.")
        raw = str(Path(settings["input_dir"]) / "MyoClass_results")
    return Path(raw).expanduser().resolve()


def validate_auto(settings,require_model=True):
    s = dict(settings)
    for key in ("input_dir", "mask_dir"):
        s[key] = str(Path(s.get(key, "")).expanduser().resolve())
        if not Path(s[key]).is_dir() or not settings.get(key):
            raise ValueError(f"Select an existing {key.replace('_', ' ')}.")
    root = project_paths(s)
    s["output_dir"] = str(root)
    if inside(s["input_dir"], root) or inside(s["mask_dir"], root):
        raise ValueError("The output folder must not equal, or contain, an input image/mask folder.")
    s["channels"] = normalize_channel_specs(s["channels"], s.get("infer_last", False))
    s.pop("infer_last", None)
    if not s["channels"]:
        raise ValueError("Enable at least one marker channel.")
    indices = [c["index"] for c in s["channels"]]
    if len(indices) != len(set(indices)) or any(i < 0 for i in indices):
        raise ValueError("Enabled channel indices must be unique and nonnegative.")
    if any(not c["name"].strip() for c in s["channels"]):
        raise ValueError("Give every enabled channel a marker/type name.")
    has_automatic = any(channel_mode(c) == "automatic" for c in s["channels"])
    if require_model and has_automatic and (not s.get("model") or not Path(s["model"]).expanduser().is_file()):
        raise ValueError("Select your custom Cellpose 3 model for the Automatic marker(s).")
    if not has_automatic:
        s["model"] = ""
    elif s.get("model"):
        s["model"] = str(Path(s["model"]).expanduser().resolve())
    for key in ("diameter", "batch_size", "pixel_size", "preview_size"):
        if not np.isfinite(s[key]) or s[key] <= 0:
            raise ValueError(f"{key.replace('_',' ')} must be positive.")
    for key in ("mask_strength", "fixed_scale", "gaussian_sigma", "flow_threshold", "min_size"):
        if not np.isfinite(s[key]) or s[key] < 0:
            raise ValueError(f"{key.replace('_',' ')} must be finite and nonnegative.")
    if not 0 < s["overlap"] <= 1:
        raise ValueError("Fiber overlap threshold must be greater than 0 and at most 1.")
    if not np.isfinite(s["cellprob_threshold"]):
        raise ValueError("Cell probability threshold must be finite.")
    return s


def read_project(root):
    root = Path(root)
    path = root / "Log" / "project.json"
    if not path.is_file():
        return {"schema": 1, "application": "MyoClass", "version": APP_VERSION, "images": {}, "active_ids": []}
    project = json.loads(path.read_text(encoding="utf-8"))
    if project.get("application") != "MyoClass" or project.get("schema") != 1:
        raise ValueError("This folder is not a supported MyoClass project.")
    return project


def save_project(root, project):
    atomic_json(Path(root) / "Log" / "project.json", project)


def image_key(path, scene, input_root):
    relative = str(Path(path).relative_to(input_root))
    stem = re.sub(r"(?i)(\.ome)?\.(tiff?|czi)$", "", Path(path).name)
    stem = re.sub(r"[^\w.\-]+", "_", stem)[:90]
    return f"{stem}__{hashlib.sha1(relative.encode('utf-8')).hexdigest()[:8]}_S{scene+1:03d}"


def source_scenes(path, s):
    if not s["all_scenes"]:
        return [0]
    if path.suffix.lower() == ".czi":
        from aicsimageio import AICSImage
        return list(range(len(AICSImage(str(path)).scenes)))
    with tiff.TiffFile(path) as source:
        return list(range(len(source.series)))


def inspect_pairs(settings):
    s = settings
    root = project_paths(s)
    image_root, mask_root = Path(s["input_dir"]), Path(s["mask_dir"])
    iterator = image_root.rglob("*") if s["recursive"] else image_root.iterdir()
    images = sorted(p for p in iterator if p.is_file() and p.suffix.lower() in (".tif", ".tiff", ".czi")
                    and not inside(p,root) and not re.search(r"(?i)_(cp_)?masks?$",p.stem))
    mask_iterator = mask_root.rglob("*") if s["recursive"] else mask_root.iterdir()
    masks = [p for p in mask_iterator if p.is_file() and p.suffix.lower() in (".tif", ".tiff")]
    # MyoSeg checkpoints retain source/scene identities even when mask names include hashes.
    indexed = {}
    checkpoint = mask_root.parent / "Log" / "checkpoints.json"
    if checkpoint.exists():
        try:
            for identifier, record in json.loads(checkpoint.read_text()).get("images", {}).items():
                if record.get("source"):
                    candidate = mask_root / f"{identifier}_masks.tif"
                    if candidate.exists():
                        indexed.setdefault((Path(record["source"]).name.casefold(),record.get("scene",0)),[]).append(candidate)
        except (ValueError, OSError):
            pass
    def base(path):
        return re.sub(r"(?i)(\.ome)?\.(tiff?|czi)$", "", path.name)
    tasks, issues = [], []
    for image in images:
        try:
            for scene in source_scenes(image,s):
                identifier = image_key(image,scene,image_root)
                exact_names = {f"{identifier}_masks".casefold()}
                direct = [p for p in masks if base(p).casefold() in exact_names]
                candidates = direct or indexed.get((image.name.casefold(),scene),[])
                if not candidates:
                    stem = base(image)
                    variants = {stem+suffix for suffix in ("", "_cp_masks", "_masks", "_mask", "_seg", "_segmentation")}
                    variants.add(f"{stem}_S{scene}_cp_masks")
                    variants.add(f"{stem}_S{scene}_masks")
                    variants.add(f"{stem}_S{scene:03d}_masks")
                    matches = [p for p in masks if base(p).casefold() in {v.casefold() for v in variants}]
                    # Generic masks without a scene suffix apply only to a single first scene.
                    if scene > 0:
                        matches = [p for p in matches if re.search(r"_S\d+",base(p),flags=re.I)]
                    candidates = matches
                candidates = sorted(set(Path(p).resolve() for p in candidates))
                if len(candidates) != 1:
                    raise ValueError(f"Scene {scene+1}: expected exactly one matching reference mask, found {len(candidates)}. Use matching image/mask stems or MyoSeg's Masks folder.")
                tasks.append({"image_path":str(image.resolve()),"mask_path":str(candidates[0]),"scene":scene,
                              "image_id":identifier,"filename":str(image.relative_to(image_root))})
        except Exception as exc:
            issues.append({"Filename":str(image.relative_to(image_root)),"Error":str(exc)})
    used = {}
    for task in tasks:
        used.setdefault(task["mask_path"],[]).append(task["image_id"])
    duplicate_masks = {path for path,ids in used.items() if len(ids)>1}
    if duplicate_masks:
        for task in tasks:
            if task["mask_path"] in duplicate_masks:
                issues.append({"Filename":task["filename"],"Error":"Reference mask matched multiple source images/scenes. Keep one source format or use unambiguous scene-specific names."})
        tasks = [task for task in tasks if task["mask_path"] not in duplicate_masks]
    return tasks, issues


def read_channels(path, scene, s):
    specs = [c for c in s["channels"] if not c.get("derived")]
    output = {}
    if Path(path).suffix.lower() == ".czi":
        from aicsimageio import AICSImage
        image = AICSImage(str(path)); image.set_scene(scene)
        for spec in specs:
            for axis,index in (("C",spec["index"]),("T",s["time"]),("Z",s["z"])):
                if index < 0 or index >= getattr(image.dims,axis):
                    raise ValueError(f"Axis {axis}: index {index} is unavailable in {image.dims}.")
        cube = image.get_image_dask_data("CYX", C=[c["index"] for c in specs], T=s["time"], Z=s["z"]).compute()
        for i,spec in enumerate(specs):
            output[f"ch{spec['index']}"] = np.asarray(cube[i])
    else:
        with tiff.TiffFile(path) as image:
            series=image.series[scene]
            array=series.asarray()
            count=array.shape[series.axes.index("C")] if "C" in series.axes else None
            for spec in specs:
                # A single-channel image is accepted only for channel 0, unlike preparation-only tools.
                plane,info=select_tiff_plane(array,series.axes,{**s,"channel":spec["index"]})
                if spec["index"] >= info["channels"]:
                    raise ValueError(f"ch{spec['index']} is unavailable ({info['channels']} channel(s)). Disable it or correct the channel layout.")
                output[f"ch{spec['index']}"]=plane
    return output


def prepare_channels(raw, reference, s):
    arrays={}
    for channel,plane in raw.items():
        if plane.ndim != 2:
            raise ValueError(f"{channel} is not a 2D plane: {plane.shape}.")
        plane=np.nan_to_num(np.asarray(plane,dtype=np.float32),nan=0.,posinf=0.,neginf=0.)
        plane=np.maximum(plane,0.)
        if plane.shape != reference.shape:
            if not s["align_to_mask"]:
                raise ValueError(f"Image {plane.shape} and reference mask {reference.shape} differ. Enable image resizing only if they cover the same field of view.")
            ratios=np.asarray(reference.shape)/np.asarray(plane.shape)
            if not np.isclose(ratios[0],ratios[1],rtol=.01,atol=2/min(plane.shape)):
                raise ValueError("Aspect ratios differ: resizing cannot align this image and mask. Check the pairing/crop.")
            plane=resize(plane,reference.shape,order=1,preserve_range=True,anti_aliasing=min(ratios)<1).astype(np.float32)
        arrays[channel]=plane
    if s["mask_mode"] == "subtract":
        if s["edge_only"]:
            labels=reference
            support=np.zeros(reference.shape,dtype=bool)
            support[:-1,:] |= labels[:-1,:] != labels[1:,:]
            support[1:,:] |= labels[1:,:] != labels[:-1,:]
            support[:,:-1] |= labels[:,:-1] != labels[:,1:]
            support[:,1:] |= labels[:,1:] != labels[:,:-1]
        else:
            support=reference > 0
        inverse=1.-support.astype(np.float32)
        if s["gaussian_sigma"]:
            inverse=gaussian_filter(inverse,s["gaussian_sigma"])
        amplitude=s["fixed_scale"] if s["scale_mode"] == "fixed" else float(np.percentile(np.stack(list(arrays.values())),99))
        for channel in arrays:
            arrays[channel]=np.maximum(arrays[channel]-inverse*(amplitude*s["mask_strength"]),0.)
    elif s["mask_mode"] == "inside":
        for channel in arrays:
            arrays[channel]=np.where(reference>0,arrays[channel],0.)
    for channel in arrays:
        if s["normalize_nonzero"]:
            arrays[channel]=normalize_nonzero_2d(arrays[channel])
    return arrays


def load_typing_model(s,emit):
    cp=installed_version("cellpose")
    if cp.split(".")[0] != "3":
        raise RuntimeError(f"Cellpose 3.x is required for this typing model; found {cp}.")
    import torch
    from cellpose import models
    cuda=torch.cuda.is_available()
    if s["device"] == "cuda" and not cuda:
        raise RuntimeError("CUDA was requested but PyTorch cannot access the GPU. Select Auto or CPU.")
    gpu=cuda and s["device"] != "cpu"
    emit(f"Loading Cellpose {cp} on {'CUDA GPU' if gpu else 'CPU'}.")
    return models.CellposeModel(gpu=gpu,pretrained_model=s["model"],model_type=None)


def predict_signal(model,plane,s):
    if not np.any(plane) or float(np.ptp(plane)) == 0:
        return np.zeros(plane.shape,dtype=np.uint32)
    result=model.eval(plane,channels=[0,0],channel_axis=None,do_3D=False,
                      diameter=s["diameter"],flow_threshold=s["flow_threshold"],
                      cellprob_threshold=s["cellprob_threshold"],min_size=s["min_size"],
                      batch_size=s["batch_size"],normalize=s["cellpose_normalize"],resample=True)
    mask=validate_mask(result[0])
    if mask.shape != plane.shape:
        raise ValueError("Cellpose returned a mask with incorrect dimensions.")
    return mask


def best_overlap(reference,prediction,label_ids,areas):
    """Maximum intersection with one predicted object / reference fiber area.

    Sparse pair counting avoids allocating max(reference_id)*max(prediction_id) bins.
    Separate fragments are not summed, matching the source notebook's rule.
    """
    if prediction.shape != reference.shape:
        raise ValueError("Predicted and reference masks must have identical dimensions.")
    selected=(reference>0)&(prediction>0)
    result=np.zeros(len(label_ids),dtype=float)
    if not selected.any():
        return result
    pairs=np.column_stack((reference[selected],prediction[selected]))
    combinations,counts=np.unique(pairs,axis=0,return_counts=True)
    indices=np.searchsorted(label_ids,combinations[:,0])
    np.maximum.at(result,indices,counts)
    return result/areas if len(areas) else result


def make_mapping(reference,predictions,s):
    regions=regionprops(reference)
    frame=pd.DataFrame([{ "label_id":int(r.label),"area_px":int(r.area),
                          "y":float(r.centroid[0]),"x":float(r.centroid[1])} for r in regions],
                       columns=["label_id","area_px","y","x"])
    labels=frame.label_id.to_numpy(dtype=np.int64)
    areas=frame.area_px.to_numpy(dtype=float)
    for spec in s["channels"]:
        name=f"ch{spec['index']}"
        if channel_mode(spec) != "automatic":
            # Manual calls start negative; inferred calls are filled after all source columns exist.
            frame[name]=np.zeros(len(frame),dtype=np.uint8)
            frame[f"overlap_{name}"]=np.nan
        else:
            overlaps=best_overlap(reference,predictions[name],labels,areas)
            frame[f"overlap_{name}"]=overlaps
            frame[name]=(overlaps>=s["overlap"]).astype(np.uint8)
    return recompute_inferred(frame, s["channels"])


def valid_entry(entry,context=None):
    if not entry.get("ready") or (context and entry.get("context") != context):
        return False
    try:
        return (file_stamp(entry["reference_path"]) == entry["reference_stamp"]
                and digest_file(entry["automatic_path"]) == entry["automatic_hash"]
                and all(Path(p).is_file() and file_stamp(p)==entry.get("prepared_stamps",{}).get(ch)
                        for ch,p in entry["prepared_paths"].items()))
    except (OSError,KeyError):
        return False


def available_entries(root):
    project=read_project(root)
    return [project["images"][key] for key in project.get("active_ids",[])
            if key in project["images"] and valid_entry(project["images"][key],project.get("active_context"))]


def save_review_session(root):
    entries=available_entries(root)
    payload={"schema":1,"application":"MyoClass","version":APP_VERSION,
             "entries":[entry for entry in entries if get_review_state(entry) not in ("Stale review","Missing review","Invalid review metadata")]}
    path=Path(root)/"review_session.json"
    atomic_json(path,payload)
    return path,payload


def run_automatic(settings,emit=None,stop_event=None,first_only=False,model_factory=None):
    emit=emit or (lambda event:None)
    stop_event=stop_event or threading.Event()
    s=validate_auto(settings)
    root=Path(s["output_dir"])
    tasks,issues=inspect_pairs(s)
    if first_only:
        tasks=tasks[:1]
    if not tasks:
        detail=issues[0]["Error"] if issues else "No source TIFF/CZI images were found."
        raise ValueError(f"No unambiguous image/mask pairs. {detail}")
    for folder in ("Reference_masks","Prepared_channels","Signal_masks","Automatic","Review","Previews","Exports","Log"):
        (root/folder).mkdir(parents=True,exist_ok=True)
    project=read_project(root)
    run_id=datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    log_path=root/"Log"/f"automatic_{run_id}.log"
    def log(text):
        with log_path.open("a",encoding="utf-8") as stream:
            stream.write(utc_now()+"  "+text+"\n")
        emit({"kind":"log","text":text})
    atomic_json(root/"Log"/f"settings_{run_id}.json",s)
    prep_keys=("axes","time","z","align_to_mask","mask_mode","edge_only","gaussian_sigma","scale_mode","fixed_scale","mask_strength","normalize_nonzero")
    predict_keys=("diameter","flow_threshold","cellprob_threshold","min_size","batch_size","cellpose_normalize")
    automatic_channels = [f"ch{c['index']}" for c in s["channels"] if channel_mode(c) == "automatic"]
    model_stamp = file_stamp(s["model"]) if automatic_channels else None
    context=signature({"input":s["input_dir"],"masks":s["mask_dir"],"channels":s["channels"],
                       "preparation":{k:s[k] for k in prep_keys},"prediction":{k:s[k] for k in predict_keys},
                       "model":model_stamp,"overlap":s["overlap"],"version":APP_VERSION})
    current_ids=[task["image_id"] for task in tasks]
    if first_only and project.get("active_context")==context:
        current_ids=list(dict.fromkeys([*project.get("active_ids",[]),*current_ids]))
    project.update({"active_context":context,"active_ids":current_ids,"settings":s,"last_errors":issues})
    save_project(root,project)
    model=None
    done=reused=0
    factory=model_factory or load_typing_model
    for index,task in enumerate(tasks):
        if stop_event.is_set():
            log("Stop requested. Completed stages and review files have been kept.")
            break
        identifier=task["image_id"]
        entry=dict(project["images"].get(identifier,{}))
        entry.update(task)
        entry.update({"context":context,"channels":s["channels"],"image_id":identifier})
        emit({"kind":"progress","current":index,"total":len(tasks),"text":task["filename"]})
        emit({"kind":"file","file":task["filename"],"state":"processing"})
        outcome = "completed"
        try:
            original_reference=validate_mask(np.squeeze(tiff.imread(task["mask_path"])))
            with tiff.TiffFile(task["mask_path"]) as source:
                native_spacing=ome_spacing(source.ome_metadata,0)
            spacing=native_spacing or (s["pixel_size"],s["pixel_size"])
            entry["native_spacing"]=list(native_spacing) if native_spacing else None
            prep_sig=signature({"source":file_stamp(task["image_path"]),"mask":file_stamp(task["mask_path"]),"scene":task["scene"],
                                "channels":[c for c in s["channels"] if not c.get("derived")],"parameters":{k:s[k] for k in prep_keys}})
            reference_path=root/"Reference_masks"/f"{identifier}_masks.tif"
            prepared_paths={f"ch{c['index']}":str(root/"Prepared_channels"/f"{identifier}_ch{c['index']}_norm.ome.tif")
                            for c in s["channels"] if not c.get("derived")}
            prep_valid=(not s["force"] and entry.get("prep_sig")==prep_sig and reference_path.exists()
                        and entry.get("reference_stamp")==file_stamp(reference_path)
                        and all(Path(p).exists() and entry.get("prepared_stamps",{}).get(ch)==file_stamp(p) for ch,p in prepared_paths.items()))
            prepared=None
            if not prep_valid:
                entry["ready"]=False
                prepared=prepare_channels(read_channels(task["image_path"],task["scene"],s),original_reference,s)
                write_image(reference_path,original_reference,spacing)
                for channel,plane in prepared.items():
                    write_image(prepared_paths[channel],plane.astype(np.float32),spacing)
                entry.update({"prep_sig":prep_sig,"prepared_stamps":{ch:file_stamp(p) for ch,p in prepared_paths.items()}})
                entry.pop("prediction_sig",None)
            entry.update({"reference_path":str(reference_path),"reference_stamp":file_stamp(reference_path),"prepared_paths":prepared_paths})
            prediction_sig=signature({"prep":prep_sig,"files":entry["prepared_stamps"],"model":model_stamp,"automatic_channels":automatic_channels,
                                      "parameters":{k:s[k] for k in predict_keys},"cellpose":installed_version("cellpose")})
            signal_paths={ch:str(root/"Signal_masks"/f"{identifier}_{ch}_signal_masks.tif") for ch in automatic_channels}
            prediction_valid=(not s["force"] and entry.get("prediction_sig")==prediction_sig
                              and all(Path(p).exists() and entry.get("signal_stamps",{}).get(ch)==file_stamp(p) for ch,p in signal_paths.items()))
            predictions={}
            if not prediction_valid:
                for channel in automatic_channels:
                    path=prepared_paths[channel]
                    plane=prepared[channel] if prepared is not None else tiff.imread(path)
                    if np.any(plane) and float(np.ptp(plane)) != 0 and model is None:
                        model=factory(s,log)
                    predictions[channel]=predict_signal(model,plane,s)
                    write_image(signal_paths[channel],predictions[channel],spacing)
                    log(f"{task['filename']} · {channel}: signal segmentation ready.")
                entry.update({"prediction_sig":prediction_sig,"signal_stamps":{ch:file_stamp(p) for ch,p in signal_paths.items()}})
            else:
                reused+=1
                log(f"{task['filename']}: reusing per-channel predictions.")
            entry["signal_paths"]=signal_paths
            map_sig=signature({"reference":entry["reference_stamp"],"predictions":entry["signal_stamps"],
                               "channels":s["channels"],"overlap":s["overlap"],"version":APP_VERSION})
            automatic_path=root/"Automatic"/f"{identifier}_mapping.xlsx"
            if not (not s["force"] and entry.get("map_sig")==map_sig and automatic_path.exists()
                    and entry.get("automatic_hash")==digest_file(automatic_path)):
                if not predictions:
                    predictions={ch:validate_mask(tiff.imread(p)) for ch,p in signal_paths.items()}
                frame=make_mapping(original_reference,predictions,s)
                save_mapping(automatic_path,frame)
            entry.update({"map_sig":map_sig,"automatic_path":str(automatic_path),"automatic_hash":digest_file(automatic_path),
                          "review_path":str(root/"Review"/f"{identifier}_mapping.xlsx"),
                          "review_meta":str(root/"Review"/f"{identifier}_review.json"),"ready":True})
            initialize_review(entry)
            manual_channels = [f"ch{c['index']}" for c in s["channels"] if channel_mode(c) == "manual"]
            if manual_channels:
                log(f"Manual marker(s): {', '.join(manual_channels)}. Check every fiber in Napari; new manual calls start negative.")
            project["images"][identifier]=entry
            save_project(root,project)
            log(f"{task['filename']}: automatic mapping ready · {get_review_state(entry)}.")
            if first_only or index in (0,len(tasks)-1):
                channel=next(iter(prepared_paths))
                plane=prepared[channel] if prepared is not None else tiff.imread(prepared_paths[channel])
                png=preview_png(plane,original_reference,s["preview_size"])
                (root/"Previews"/f"{identifier}.png").write_bytes(png)
                emit({"kind":"preview","png":png,"text":f"{task['filename']} · scene {task['scene']+1} · prepared {channel} and reference fiber boundaries. Review marker calls in Napari."})
            done+=1
        except Exception as exc:
            entry["ready"]=False
            project["images"][identifier]=entry
            outcome = "failed"
            issues.append({"Filename":task["filename"],"Error":str(exc)})
            log(f"ERROR {task['filename']}: {exc}")
            with log_path.open("a",encoding="utf-8") as stream:
                stream.write(traceback.format_exc()+"\n")
            save_project(root,project)
        emit({"kind":"file","file":task["filename"],"state":outcome})
        emit({"kind":"progress","current":index+1,"total":len(tasks),"text":f"Completed {index+1} / {len(tasks)} image(s)/scene(s)"})
    project["last_errors"]=issues
    save_project(root,project)
    atomic_csv(root/"Log"/"automatic_errors.csv",pd.DataFrame(issues,columns=["Filename","Error"]))
    session,payload=save_review_session(root)
    return {"successful":done,"errors":len(issues),"reused":reused,"stopped":stop_event.is_set(),
            "output_dir":str(root),"reviewable":len(payload["entries"]),"log_path":str(log_path)}


def selected_preview(settings):
    s=validate_auto(settings,require_model=False)
    tasks,issues=inspect_pairs(s)
    if not tasks:
        raise ValueError(issues[0]["Error"] if issues else "No matching images and masks.")
    task=tasks[0]
    mask=validate_mask(np.squeeze(tiff.imread(task["mask_path"])))
    channels=prepare_channels(read_channels(task["image_path"],task["scene"],s),mask,s)
    channel=next(iter(channels))
    return {"kind":"preview","png":preview_png(channels[channel],mask,s["preview_size"]),
            "text":f"{task['filename']} · {channel} · {mask.shape[1]} × {mask.shape[0]} px. {len(tasks)} matched image(s)/scene(s); {len(issues)} pairing issue(s)."}


def validate_export(settings):
    for name in ("pixel_size","csa_max","csa_step","feret_max","feret_step"):
        if not np.isfinite(settings[name]) or settings[name] <= 0:
            raise ValueError(f"{name.replace('_',' ')} must be positive and finite.")
    if any(not np.isfinite(settings[k]) or settings[k]<0 for k in ("area_min","area_max")):
        raise ValueError("CSA filter bounds must be nonnegative.")
    if settings["area_max"] and settings["area_max"] <= settings["area_min"]:
        raise ValueError("Maximum CSA must exceed minimum CSA, or be 0 to disable it.")
    for prefix in ("csa","feret"):
        ratio=settings[f"{prefix}_max"]/settings[f"{prefix}_step"]
        if ratio>500 or not np.isclose(ratio,round(ratio)):
            raise ValueError(f"{prefix.upper()} distribution maximum must be a multiple of the bin width (at most 500 bins).")


def export_measurements(settings,emit=None):
    emit=emit or (lambda event:None)
    validate_export(settings)
    root=project_paths(settings)
    project=read_project(root)
    entries=available_entries(root)
    if not entries:
        raise ValueError("No valid automatic mappings found. Load the output folder or run classification first.")
    rows,issues=[],[]
    for i,entry in enumerate(entries):
        emit({"kind":"progress","current":i,"total":len(entries),"text":f"Measuring {entry['filename']}"})
        try:
            state=get_review_state(entry)
            if settings["export_source"]=="reviewed" and state!="Reviewed":
                issues.append({"Filename":entry["filename"],"Reason":f"Not exported: {state}"})
                continue
            automatic=load_mapping(entry["automatic_path"],entry)
            frame=automatic if settings["export_source"]=="automatic" else load_mapping(entry["review_path"],entry,automatic)
            frame=recompute_inferred(frame.copy(),entry["channels"])
            mask=validate_mask(tiff.imread(entry["reference_path"]))
            native=entry.get("native_spacing")
            spacing=native if settings["calibration"]=="metadata" and native else [settings["pixel_size"]]*2
            region_map={int(r.label):r for r in regionprops(mask)}
            if set(region_map)!=set(frame.label_id):
                raise ValueError("Mapping label IDs differ from the reference mask.")
            image_rows=[]
            for _,item in frame.iterrows():
                region=region_map[int(item.label_id)]
                area=float(region.area*spacing[0]*spacing[1])
                border=(region.bbox[0]==0 or region.bbox[1]==0 or region.bbox[2]==mask.shape[0] or region.bbox[3]==mask.shape[1])
                reasons=[]
                if settings["exclude_border"] and border: reasons.append("Image border")
                if area<settings["area_min"]: reasons.append("Below minimum CSA")
                if settings["area_max"] and area>settings["area_max"]: reasons.append("Above maximum CSA")
                calls={f"ch{c['index']}":int(item[f"ch{c['index']}"]) for c in entry["channels"]}
                combination="+".join(ch for ch,value in calls.items() if value) or "All negative"
                image_rows.append({"Image_ID":entry["image_id"],"Filename":entry["filename"],"Scene":entry["scene"]+1,
                             "label_id":int(region.label),"Region_Label":int(region.label),"area_px":int(region.area),"y":float(region.centroid[0]),"x":float(region.centroid[1]),
                             "Area_µm²":area,"Feret_Diameter_µm":minimum_feret(region,spacing) if settings["compute_feret"] else np.nan,
                             **calls,"Combination":combination,"Included":not reasons,"Exclusion_reason":"; ".join(reasons),
                             "Classification_source":"Reviewed" if settings["export_source"]=="reviewed" else "Automatic",
                             "Pixel_size_Y_µm":spacing[0],"Pixel_size_X_µm":spacing[1]})
            rows.extend(image_rows)
            # An empty mask still gets a summary row through the image list below.
            entry["export_ok"]=True
        except Exception as exc:
            issues.append({"Filename":entry["filename"],"Reason":str(exc)})
    successful=[e for e in entries if e.get("export_ok")]
    if not successful:
        raise ValueError("No images eligible for export. Mark at least one image reviewed in Napari, or select Automatic predictions. Details: "+str(issues[:3]))
    channels=successful[0]["channels"]
    channel_names=[f"ch{c['index']}" for c in channels]
    columns=["Image_ID","Filename","Scene","label_id","Region_Label","area_px","y","x","Area_µm²","Feret_Diameter_µm",
             *channel_names,"Combination","Included","Exclusion_reason","Classification_source","Pixel_size_Y_µm","Pixel_size_X_µm"]
    details=pd.DataFrame(rows,columns=columns)
    included=details.loc[details.Included.astype(bool)]
    sheets={"Detailed_Labels_all":details}
    scopes=[("all",None)]+[(ch,ch) for ch in channel_names]
    for scope,channel in scopes:
        subset=included if channel is None else included.loc[included[channel]==1]
        summaries=[]; ccounts=[]; cpcts=[]; fcounts=[]; fpcts=[]
        for entry in successful:
            current=subset.loc[subset.Image_ID==entry["image_id"]]
            all_image=included.loc[included.Image_ID==entry["image_id"]]
            identity={"Image_ID":entry["image_id"],"Filename":entry["filename"],"Scene":entry["scene"]+1}
            areas=current['Area_µm²'].to_numpy(dtype=float); ferets=current['Feret_Diameter_µm'].to_numpy(dtype=float)
            def stat(values,func):
                values=values[np.isfinite(values)]
                return float(func(values)) if len(values) else np.nan
            summaries.append({**identity,"Number_of_Labels":len(current),"Percent_of_all_fibers":len(current)/len(all_image)*100 if len(all_image) else np.nan,
                              "Mean_CSA_µm²":stat(areas,np.mean),"Median_CSA_µm²":stat(areas,np.median),
                              "Min_CSA_µm²":stat(areas,np.min),"Max_CSA_µm²":stat(areas,np.max),
                              "Mean_Feret_Diameter_µm":stat(ferets,np.mean)})
            c,p=distribution(areas,settings['csa_max'],settings['csa_step'],'µm²'); ccounts.append({**identity,**c}); cpcts.append({**identity,**p})
            c,p=distribution(ferets,settings['feret_max'],settings['feret_step'],'µm'); fcounts.append({**identity,**c}); fpcts.append({**identity,**p})
        sheets.update({f"Summary_{scope}":pd.DataFrame(summaries),f"CSA_Distribution_{scope}":pd.DataFrame(ccounts),
                       f"CSA_Distribution%_{scope}":pd.DataFrame(cpcts),f"Feret_Distribution_{scope}":pd.DataFrame(fcounts),
                       f"Feret_Distribution%_{scope}":pd.DataFrame(fpcts)})
        if channel is not None:
            sheets[f"Detailed_Labels_{scope}"]=subset
    combinations=[]
    for entry in successful:
        frame=included.loc[included.Image_ID==entry['image_id']]
        for combination,count in frame.Combination.value_counts().items():
            combinations.append({"Filename":entry['filename'],"Scene":entry['scene']+1,"Combination":combination,
                                 "Number_of_Labels":int(count),"Percent_of_all_fibers":count/len(frame)*100})
    sheets["Combinations"]=pd.DataFrame(combinations,columns=["Filename","Scene","Combination","Number_of_Labels","Percent_of_all_fibers"])
    origins={"automatic":"Cellpose fluorescence prediction (editable in review)","manual":"Manual in Napari; initialized negative", "inferred":"Inferred from all other enabled markers; recalculated at export"}
    sheets["Marker_names"]=pd.DataFrame([{ "Channel":f"ch{c['index']}","Name":c['name'],"Mode":channel_mode(c),"Origin":origins[channel_mode(c)]} for c in channels])
    pending=[{"Filename":project['images'].get(key,{}).get('filename',key),"Reason":"Automatic classification pending, failed or outdated"}
             for key in project.get('active_ids',[]) if key not in {e['image_id'] for e in entries}]
    issues.extend(pending)
    issues.extend({"Filename":error["Filename"],"Reason":error["Error"]} for error in project.get("last_errors",[]))
    sheets['Export_issues']=pd.DataFrame(issues,columns=['Filename','Reason'])
    sheets['Parameters']=pd.DataFrame([{'Parameter':k,'Value':str(v)} for k,v in settings.items()])
    folder=root/'Exports'/settings['export_source']
    folder.mkdir(parents=True,exist_ok=True)
    path=folder/'segmentation_results_by_type.xlsx'
    if path.exists():
        backup=root/'Log'/'Backups'/f"{settings['export_source']}_{datetime.now().strftime('%Y%m%d_%H%M%S_%f')}.xlsx"
        backup.parent.mkdir(parents=True,exist_ok=True); shutil.copy2(path,backup)
    temp=folder/f'.export_{uuid.uuid4().hex}.xlsx'
    try:
        with pd.ExcelWriter(temp,engine='openpyxl') as writer:
            for name,frame in sheets.items():
                if len(frame)>1_048_575:
                    raise ValueError('An Excel sheet exceeds the row limit. Export fewer images per project.')
                frame.to_excel(writer,sheet_name=name,index=False)
        os.replace(temp,path)
    finally:
        temp.unlink(missing_ok=True)
    atomic_csv(folder/'Detailed_Labels_all.csv',details)
    atomic_csv(folder/'Summary_all.csv',sheets['Summary_all'])
    atomic_csv(folder/'Combinations.csv',sheets['Combinations'])
    atomic_csv(folder/'Export_issues.csv',sheets['Export_issues'])
    emit({'kind':'progress','current':len(entries),'total':len(entries),'text':'Export complete'})
    return {'workbook':str(path),'images':len(successful),'issues':len(issues),'fibers':len(included)}


def napari_child_environment(executable):
    """Let the selected interpreter discover its own Qt/Python packages.

    QtPy can set QT_API in the parent process. Passing that choice to a
    different environment can hide an otherwise installed Qt binding.
    OpenCV can also set a plugin path for its own bundled Qt libraries.
    """
    environment=os.environ.copy()
    for name in ('QT_API','QT_QPA_PLATFORM_PLUGIN_PATH','QT_PLUGIN_PATH','QT_QPA_FONTDIR',
                 'PYTHONPATH','PYTHONHOME','PYTHONEXECUTABLE','__PYVENV_LAUNCHER__'):
        environment.pop(name,None)
    environment['PATH']=str(Path(executable).parent)+os.pathsep+environment.get('PATH','')
    environment['PYTHONUNBUFFERED']='1'
    return environment


def check_napari_environment(executable,environment,review_path):
    probe='''import sys
print("Python: " + sys.executable, flush=True)
import napari
print("Napari: " + napari.__version__, flush=True)
from qtpy import API_NAME, QtCore, QtWidgets
print("Qt binding: " + API_NAME + " / Qt " + QtCore.qVersion(), flush=True)
import pandas as pd
import tifffile
pd.read_excel(sys.argv[1], sheet_name="mapping", nrows=0, engine="openpyxl")
print("Review workbook dependencies: OK", flush=True)
'''
    try:
        result=subprocess.run([executable,'-c',probe,str(review_path)],env=environment,
                              capture_output=True,text=True,timeout=30)
    except subprocess.TimeoutExpired as exc:
        raise RuntimeError(f'Napari startup check timed out for {executable}. Test this Python environment in a terminal.') from exc
    if result.returncode:
        details='\n'.join((result.stdout+'\n'+result.stderr).splitlines()[-22:])
        hint=('A Qt binding is unavailable in this selected environment (conda-forge package: pyqt). '
              if 'QtBindingsNotFoundError' in details else '')
        raise RuntimeError(f'Napari startup check failed for {executable}. {hint}\n{details}')
    return result.stdout


def launch_napari_project(root,image_id,script_source,python_executable=None):
    root=Path(root)
    session,payload=save_review_session(root)
    if not any(e['image_id']==image_id for e in payload['entries']):
        raise ValueError('Selected review is unavailable or stale. Reset it from automatic predictions before opening Napari.')
    if sys.platform.startswith('linux') and not (os.environ.get('DISPLAY') or os.environ.get('WAYLAND_DISPLAY')):
        raise RuntimeError('Napari needs a desktop display on the computer running Voilà. Start Voilà from your desktop session, or open review_session.json with the standalone correction script.')
    executable=python_executable.strip() if python_executable else sys.executable
    executable=str(Path(executable).expanduser().absolute())
    if not Path(executable).is_file():
        raise ValueError('Napari Python executable was not found.')
    environment=napari_child_environment(executable)
    entry=next(e for e in payload['entries'] if e['image_id']==image_id)
    startup=check_napari_environment(executable,environment,entry['review_path'])
    tools_folder=root/'Log'/'Tools'; tools_folder.mkdir(parents=True,exist_ok=True)
    script=tools_folder/'Corrections_napari_MyoClass.py'
    script.write_text(script_source,encoding='utf-8')
    log_path=root/'Log'/f"napari_{datetime.now().strftime('%Y%m%d_%H%M%S_%f')}.log"
    with log_path.open('w',encoding='utf-8') as stream:
        stream.write(startup+'\n'); stream.flush()
        process=subprocess.Popen([executable,str(script),'--session',str(session),'--image-id',image_id],
                                 stdout=stream,stderr=subprocess.STDOUT,cwd=str(root),env=environment)
    return process,log_path


In [ ]:
from IPython.display import HTML, Markdown, display

display(HTML(r'''
<style>
:root { color-scheme: light; }
body { background: #faf5f5 !important; }
#rendered_cells {
  position: relative !important;
  inset: auto !important;
  width: min(100%, 1150px) !important;
  max-width: 1150px !important;
  margin: 0 auto !important;
  padding: 34px 20px 68px !important;
  display: block !important;
  text-align: left !important;
  box-sizing: border-box !important;
}
.myoclass-app { width: 100%; min-width: 0; gap: 18px; color: #352529; font-family: Inter, -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif; }
.myoclass-app * { box-sizing: border-box; }
.mc-hero {
  position: relative; overflow: hidden; padding: 34px 42px 30px;
  border-radius: 20px; color: #fff;
  background: radial-gradient(circle at 88% 10%, #ba5263 0, transparent 31%), linear-gradient(125deg, #310b13 0%, #6c1426 67%, #9a2039 100%);
  box-shadow: 0 18px 40px rgba(90, 15, 35, .16);
}
.mc-hero::after { content: ""; position: absolute; right: -100px; top: 35px; width: 290px; height: 290px; border: 1px solid rgba(255,255,255,.14); border-radius: 50%; pointer-events: none; }
.mc-hero-top { display: flex; align-items: center; justify-content: space-between; gap: 16px; }
.mc-mark { width: 43px; height: 43px; display: inline-flex; align-items: center; justify-content: center; background: #fecdd3; color: #6b1427; border-radius: 12px; font-size: 15px; font-weight: 800; letter-spacing: -.03em; }
.mc-version { color: #fbcbd3; font-size: 11px; font-weight: 700; letter-spacing: .15em; }
.mc-hero h1 { margin: 22px 0 7px; color: #fff; font-size: clamp(34px, 5vw, 52px); line-height: 1.05; font-weight: 760; letter-spacing: -.045em; }
.mc-hero p { margin: 0; max-width: 620px; color: #fbe3e7; font-size: 15px; line-height: 1.6; }
.mc-hero-foot { display: flex; flex-wrap: wrap; gap: 11px; margin-top: 26px; }
.mc-hero-foot span { padding: 7px 12px; border-radius: 999px; background: rgba(255,255,255,.1); border: 1px solid rgba(255,255,255,.12); color: #fff0f2; font-size: 11px; font-weight: 650; }
.mc-hero-foot .mc-paper-link { display: inline-flex; align-items: center; margin-left: auto; padding: 7px 13px; border-radius: 999px; background: #fecdd3; color: #64182a; font-size: 11px; font-weight: 750; text-decoration: none; }
.mc-hero-foot .mc-paper-link:hover, .mc-hero-foot .mc-paper-link:focus { background: #ffe4e8; color: #64182a; text-decoration: underline; }
.mc-card { width: 100%; min-width: 0; padding: 26px 30px 28px; background: #fff; border: 1px solid #eddddd; border-radius: 17px; box-shadow: 0 5px 22px rgba(90, 15, 35, .045); gap: 15px; }
.mc-section-heading { display: flex; gap: 15px; align-items: flex-start; padding-bottom: 16px; border-bottom: 1px solid #f4e9ea; }
.mc-number { flex: none; width: 38px; height: 38px; border-radius: 11px; display: inline-flex; align-items: center; justify-content: center; background: #fff0f1; color: #b4233a; font-size: 12px; font-weight: 780; letter-spacing: .04em; }
.mc-section-heading h2 { margin: 0 0 4px; color: #42272e; font-size: 19px; line-height: 1.25; font-weight: 750; letter-spacing: -.02em; }
.mc-section-heading p { margin: 0; color: #7f656c; font-size: 13px; line-height: 1.5; }
.mc-field-label { margin: 4px 0 8px; display: flex; flex-direction: column; gap: 3px; }
.mc-field-label strong { font-size: 13px; color: #4d3038; }
.mc-field-label span { color: #7f656c; font-size: 12px; line-height: 1.45; }
.mc-chooser { min-width: 0; max-width: 100%; }
.mc-chooser select, .mc-chooser input { max-width: 100%; }
.myoclass-app .widget-label { color: #594149; font-size: 13px; font-weight: 550; }
.myoclass-app input, .myoclass-app select { border-color: #dfcdd1; border-radius: 7px; }
.myoclass-app input:focus, .myoclass-app select:focus { border-color: #bd2945; outline-color: #bd2945; }
.mc-buttons { gap: 10px; margin: 2px 0 0; }
.mc-buttons .widget-button, .mc-buttons button { border-radius: 9px; font-size: 13px; font-weight: 700; box-shadow: none; transition: transform .15s ease, box-shadow .15s ease; }
.mc-buttons .widget-button:hover, .mc-buttons button:hover { transform: translateY(-1px); box-shadow: 0 5px 15px rgba(120, 25, 50, .17); }
.mc-primary-button, .mc-primary-button button { background: #b4233a !important; border-color: #b4233a !important; color: #fff !important; }
.mc-primary-button:hover, .mc-primary-button button:hover { background: #941e32 !important; }
.mc-secondary-button, .mc-secondary-button button { background: #fff0f2 !important; border: 1px solid #f2c7cf !important; color: #941e32 !important; }
.mc-status { display: inline-flex; align-items: center; gap: 9px; padding: 9px 12px; border-radius: 8px; font-size: 12px; font-weight: 650; }
.mc-status-dot { display: inline-block; width: 8px; height: 8px; border-radius: 50%; background: currentColor; }
.mc-ready { background: #f5eded; color: #7d656c; }
.mc-running { background: #fff0f1; color: #b4233a; }
.mc-complete { background: #fdebef; color: #941e32; }
.mc-error { background: #fff0eb; color: #ad4c35; }
.mc-output { width: 100%; min-width: 0; padding: 3px 2px; }
.mc-output .widget-progress { width: min(100%, 520px) !important; }
.mc-output .progress-bar { background: #bd2945 !important; }
.mc-output .jp-OutputArea pre, .mc-output .output_subarea pre { text-align: left !important; white-space: pre-wrap; overflow-wrap: anywhere; font-size: 12px; line-height: 1.55; }
.mc-output h2 { font-size: 15px; color: #7b293c; }
@media (max-width: 640px) {
  #rendered_cells { padding: 14px 12px 40px !important; }
  .mc-hero { padding: 25px 23px; }
  .mc-card { padding: 20px 18px; }
  .mc-version { font-size: 9px; letter-spacing: .08em; }
  .mc-hero-foot .mc-paper-link { margin-left: 0; }
  .mc-buttons > * { width: 100% !important; }
}

.myoclass-app .widget-html-content { width: 100%; }
.myoclass-app .widget-checkbox { height: auto; min-height: 29px; }
.myoclass-app .widget-checkbox label { white-space: normal; height: auto; line-height: 1.5; }
.mc-card > .widget-box { min-width: 0; }
.mc-note { padding: 10px 13px; background: #fcf6f7; color: #7b5d65; border-radius: 9px; font-size: 12px; line-height: 1.6; overflow-wrap: anywhere; }
.mc-help, .mc-output-map { color: #7b646b; font-size: 12px; line-height: 1.9; }
.mc-output-map strong, .mc-help strong { color: #723847; }
.mc-log { padding: 12px 15px; border: 1px solid #f1e4e7; border-radius: 10px; font-family: ui-monospace, SFMono-Regular, monospace; font-size: 11px; line-height: 1.75; color: #866a73; overflow-wrap: anywhere; }
.mc-log > div + div { border-top: 1px solid #fcf6f7; padding-top: 3px; margin-top: 3px; }
.mc-preview { border-radius: 10px; object-fit: contain; max-height: 650px; background: #fcf8f9; }
.mc-progress .progress { height: 8px; background: #fff0f2; border-radius: 8px; }
.mc-progress .progress-bar { background: #bd2945; }
.mc-footer { padding: 4px 12px 18px; color: #9b7d87; font-size: 11px; line-height: 1.8; }
.myoclass-app .widget-button:disabled { opacity: .45; }
@media (max-width: 640px) {
  .myoclass-app .widget-inline-hbox { flex-wrap: wrap; height: auto; gap: 4px; }
  .myoclass-app .widget-label, .myoclass-app .widget-inline-hbox > label { width: 100% !important; text-align: left !important; margin-left: 0; }
}

.myoclass-app input[type="checkbox"] { accent-color: #b4233a; }
.mc-busy .mc-chooser { pointer-events: none; opacity: .55; }
.mc-note pre { white-space: pre-wrap; overflow-wrap: anywhere; font-size: 11px; }
.mc-files{max-height:260px;overflow:auto;border:1px solid #ebdfe4;border-radius:12px;background:white}.mc-file{padding:7px 12px;display:flex;justify-content:space-between;gap:12px;border-bottom:1px solid #f2edf0;font-size:12px}.mc-file span{overflow:hidden;text-overflow:ellipsis;white-space:nowrap}.mc-file b{color:#8057e8;text-transform:capitalize}.mc-progress .progress{border-radius:12px!important;overflow:hidden!important}.mc-progress .progress-bar{background:#8057e8!important;border-radius:12px!important}
</style>
'''))


In [ ]:
import ipywidgets as widgets
from ipyfilechooser import FileChooser

def default_model_file(name):
    """Use a model downloaded by MyoTools, when it is present."""
    import os
    folder = Path(os.environ.get("MYOTOOLS_MODEL_DIR", str(Path.cwd() / "Hub" / "runtime" / "models")))
    model = folder / name
    return model if model.is_file() else None


def default_image_directory():
    """Use the user's real Desktop, or their home if no Desktop is available.

    This standard-library-only function is embedded in the notebooks so they
    also work when launched directly, without importing or starting the hub.
    """
    import os
    import subprocess
    from pathlib import Path

    if os.name == "nt":
        command = ["powershell.exe", "-NoProfile", "-Command",
                   "[Console]::OutputEncoding = [System.Text.Encoding]::UTF8; [Environment]::GetFolderPath('DesktopDirectory')"]
    else:
        command = ["xdg-user-dir", "DESKTOP"]
    try:
        result = subprocess.run(command, capture_output=True, text=True, timeout=5,
                                encoding="utf-8", errors="replace")
        if result.returncode == 0 and result.stdout.strip():
            desktop = Path(result.stdout.strip().lstrip("\ufeff")).expanduser()
            if desktop.is_absolute() and desktop.is_dir():
                return desktop
    except (OSError, subprocess.SubprocessError):
        pass
    home = Path.home()
    return next((path for path in (home / "Desktop", home / "Bureau") if path.is_dir()), home)

from IPython.display import HTML, display


def field_label(title,description=""):
    return widgets.HTML(value=f'<div class="mc-field-label"><strong>{html.escape(title)}</strong><span>{html.escape(description)}</span></div>')


def card(number,title,description,children):
    heading=widgets.HTML(value=f'<div class="mc-section-heading"><span class="mc-number">{number}</span><div><h2>{html.escape(title)}</h2><p>{html.escape(description)}</p></div></div>')
    box=widgets.VBox([heading,*children],layout=widgets.Layout(width="100%"))
    box.add_class('mc-card')
    return box


class MyoClassApp:
    def __init__(self):
        self.busy=False
        self.task=None
        self.stop_event=threading.Event()
        self.napari_process=None
        self.napari_watch_task=None
        self.controls=[]
        self.action_buttons=[]
        self.marker_rows=[]
        self.messages=[]
        self.style={'description_width':'215px'}
        self.header=widgets.HTML(value=f'''<div class="mc-hero">
            <div class="mc-hero-top"><span class="mc-mark">MC</span><span class="mc-version">VERSION {APP_VERSION} · CELLPOSE 3</span></div>
            <h1>MyoClass</h1><p>Type muscle fibers from immunofluorescence.<br>Predict marker positivity, review calls in Napari and export fiber morphometry.</p>
            <div class="mc-hero-foot"><span>01 &nbsp;Classify</span><span>02 &nbsp;Review</span><span>03 &nbsp;Measure</span></div></div>''')
        self.input_chooser=self.chooser(directories=True,path=default_image_directory())
        self.mask_chooser=self.chooser(directories=True)
        installed_model=default_model_file("MyoClass")
        self.model_chooser=self.chooser(path=installed_model.parent if installed_model else None,
                                        filename=installed_model.name if installed_model else "",
                                        select_default=bool(installed_model))
        self.input_box=widgets.VBox([field_label('Fluorescence image folder','Original multichannel TIFF, OME-TIFF or CZI images.'),self.input_chooser])
        self.mask_box=widgets.VBox([field_label('Reference fiber masks','Select MyoSeg’s Masks folder, or integer TIFF masks with matching image stems.'),self.mask_chooser])
        self.model_box=widgets.VBox([field_label('Cellpose 3 typing model','The custom model detecting positive fibers in each immunofluorescence channel.'),self.model_chooser])
        self.output_path=widgets.Text(placeholder='Automatic: <image folder>/MyoClass_results',layout=widgets.Layout(width='100%'))
        self.controls.append(self.output_path)
        self.output_chooser=self.chooser(directories=True)
        self.output_chooser.register_callback(lambda c:setattr(self.output_path,'value',c.selected or ''))
        self.recursive=self.check('Include subfolders',False)
        self.load_button=self.button('Load existing project','folder-open','secondary',lambda:self.start('load'))
        files=card('01','Choose files','Reference fiber masks define the objects to classify. Their label IDs are preserved throughout the workflow.',[
            self.input_box,self.mask_box,self.model_box,field_label('Output / existing project folder','Leave blank for a new MyoClass_results folder, or select an existing project to continue its review.'),
            self.output_path,self.output_chooser,self.recursive,self.load_button])

        self.marker_box=widgets.VBox(layout=widgets.Layout(width='100%',gap='10px'))
        for i,mode in enumerate(('manual','automatic','inferred')):
            self.add_marker(index=i,name=f'Marker {chr(65+i)}',mode=mode)
        self.add_marker_button=self.button('Add marker channel','plus','secondary',lambda:self.add_marker())
        self.axes=self.control(widgets.Dropdown,'TIFF channel layout','auto',options=[('Auto (TIFF/OME axes)','auto'),('Channels first: C × Y × X','CYX'),('Channels last: Y × X × C','YXC')])
        self.all_scenes=self.check('Process all CZI scenes / TIFF series',True)
        self.time=self.control(widgets.BoundedIntText,'Time point',0,min=0,max=99999)
        self.z=self.control(widgets.BoundedIntText,'Z plane',0,min=0,max=99999)
        self.align_to_mask=self.check('Resize fluorescence images to the reference mask dimensions',False)
        self.mask_mode=self.control(widgets.Dropdown,'Background handling','subtract',options=[('Subtract outside reference fibers','subtract'),('Set outside fibers to zero','inside'),('No background masking','none')])
        self.edge_only=self.check('Use reference label edges as support before inversion',False)
        self.scale_mode=self.control(widgets.Dropdown,'Subtraction amplitude','p99',options=[('99th percentile across measured channels','p99'),('Fixed intensity','fixed')])
        self.mask_strength=self.control(widgets.BoundedFloatText,'Subtraction strength',1.,min=0,max=100,step=.1)
        self.fixed_scale=self.control(widgets.BoundedFloatText,'Fixed intensity',2000.,min=0,max=1e9,step=100.)
        self.gaussian_sigma=self.control(widgets.BoundedFloatText,'Mask smoothing (px)',0.,min=0,max=100,step=.5)
        self.normalize_nonzero=self.check('Normalize positive intensities; keep zero background',True)
        preparation=card('02','Choose markers and prepare channels','Channel numbering starts at 0. Edit channel names.',[
            self.marker_box,self.add_marker_button,
            field_label('Choose each marker’s mode','Automatic: send fluorescence to Cellpose, then review in Napari. Manual: display fluorescence in Napari and start calls at 0, without Cellpose. Inferred: positive only when all other enabled markers are negative; updated after corrections and at export. Only one inferred category is allowed.'),
            field_label('Independent source markers','Automatic and manual calls can both be positive for the same fiber, preserving hybrids. Check every manual marker before marking the image reviewed. Inferred calls remain read-only and need no fluorescence channel.'),
            self.axes,self.all_scenes,self.time,self.z,self.align_to_mask,
            field_label('Alignment','Only enable resizing when images and masks cover the same field of view. Resizing does not register shifted or differently cropped images.'),
            self.mask_mode,self.edge_only,self.scale_mode,self.mask_strength,self.fixed_scale,self.gaussian_sigma,self.normalize_nonzero])

        self.diameter=self.control(widgets.BoundedFloatText,'Fiber diameter (px)',70.,min=.1,max=1e6,step=5.)
        self.device=self.control(widgets.Dropdown,'Processing device','auto',options=[('Auto: CUDA if available, otherwise CPU','auto'),('CPU','cpu'),('CUDA GPU (required)','cuda')])
        self.diameter_guide_crop=self.control(widgets.Dropdown,'Diameter guide crop',512,options=[256,384,512,768,1024])
        self.diameter_guide_button=self.button('Preview diameter guide','circle-o','secondary',self.show_diameter_guide)
        self.diameter_guide_image=widgets.Image(format='png',layout=widgets.Layout(width='100%',max_width='520px'))
        self.diameter_guide_message=widgets.HTML(value='<div class="mc-note">Yellow circle indicates the Cellpose diameter on the prepared fluorescence channel.</div>')
        self.flow_threshold=self.control(widgets.BoundedFloatText,'Flow threshold',.4,min=0,max=100,step=.1)
        self.cellprob_threshold=self.control(widgets.FloatText,'Cell probability threshold',0.)
        self.min_size=self.control(widgets.BoundedIntText,'Minimum object size (px²)',15,min=0,max=100000000)
        self.batch_size=self.control(widgets.BoundedIntText,'Cellpose batch size',8,min=1,max=1024)
        self.cellpose_normalize=self.check('Use Cellpose percentile normalization',True)
        self.overlap=self.control(widgets.BoundedFloatText,'Positive overlap fraction',.5,min=.001,max=1.,step=.05)
        self.force=self.check('Recompute automatic outputs (existing manual edits are preserved)',False)
        self.pairs_button=self.button('Check image/mask pairs','link','secondary',lambda:self.start('pairs'))
        self.preview_button=self.button('Preview prepared input','image','secondary',lambda:self.start('preview'))
        self.test_button=self.button('Test first image','flask','secondary',lambda:self.start('test'))
        self.run_button=self.button('Run / resume typing','play','primary',lambda:self.start('run'))
        self.stop_button=widgets.Button(description='Stop after image',icon='stop',layout=widgets.Layout(width='215px',height='44px'),disabled=True)
        self.stop_button.add_class('mc-secondary-button')
        self.stop_button.on_click(self.stop)
        action_bar=widgets.HBox([self.pairs_button,self.preview_button,self.test_button,self.run_button,self.stop_button],layout=widgets.Layout(width='100%',flex_flow='row wrap'))
        action_bar.add_class('mc-buttons')
        self.status=widgets.HTML(value=self.status_html('ready','Ready to classify'))
        self.progress=widgets.FloatProgress(value=0,min=0,max=1,layout=widgets.Layout(width='100%',height='12px'))
        self.progress.add_class('mc-progress')
        self.progress_label=widgets.HTML()
        self.log=widgets.HTML()
        self.file_status=widgets.HTML(value="<div class=\"mc-note\">Processed files will appear here.</div>")
        self.file_states={}
        self.preview_image=widgets.Image(format='png',layout=widgets.Layout(width='100%',max_width='100%',max_height='650px',display='none'))
        self.preview_image.add_class('mc-preview')
        self.preview_caption=widgets.HTML()
        automatic=card('03','Run / prepare typing','Only Automatic markers are sent to Cellpose. Manual fluorescence is prepared for Napari; no model is required when every source marker is Manual.',[
            self.diameter,self.diameter_guide_crop,self.diameter_guide_button,self.diameter_guide_message,self.diameter_guide_image,self.device,self.flow_threshold,self.cellprob_threshold,self.min_size,self.batch_size,self.cellpose_normalize,self.overlap,
            field_label('Overlap is not a probability','The default 0.5 means at least 50% of the reference fiber is covered by one predicted object. Separate fragments are not added together.'),
            self.force,action_bar,self.status,self.progress,self.progress_label,self.file_status,self.log,self.preview_image,self.preview_caption])

        self.review_image=widgets.Dropdown(options=[('Run typing or load a project first','')],layout=widgets.Layout(width='100%'))
        self.review_summary=widgets.HTML()
        self.napari_python=widgets.Text(placeholder='Optional: Python executable of a separate Napari environment',layout=widgets.Layout(width='100%'))
        self.controls.append(self.napari_python)
        self.open_napari=self.button('Open Napari review','external-link','primary',self.open_editor)
        self.refresh_review=self.button('Refresh review status','refresh','secondary',self.refresh_project)
        self.reset_review=self.button('Reset selected review','undo','secondary',self.reset_selected)
        review_buttons=widgets.HBox([self.open_napari,self.refresh_review,self.reset_review],layout=widgets.Layout(width='100%',flex_flow='row wrap'))
        review_buttons.add_class('mc-buttons')
        self.review_message=widgets.HTML(value='<div class="mc-note">Napari opens as a separate desktop window on the computer running Voilà. Save and close it before exporting reviewed results.</div>')
        review=card('04','Review marker calls in Napari','F1 = negative, F2 = positive for Automatic / Manual markers; Ctrl+S saves a draft. The Inferred category updates automatically.',[
            self.review_summary,self.review_image,review_buttons,
            field_label('Review states','Drafts stay editable. Mark image reviewed in Napari after checking all markers. Reset replaces the selected draft from automatic calls and backs up previous edits.'),
            self.review_message,field_label('Napari Python (optional)','Leave blank to use the same Python environment as this notebook.'),self.napari_python])

        self.calibration=self.control(widgets.Dropdown,'Mask pixel calibration','metadata',options=[('Reference mask metadata; manual fallback','metadata'),('Manual (mask resolution)','manual')])
        self.pixel_size=self.control(widgets.BoundedFloatText,'Mask pixel size (µm)',.6928,min=.000001,max=1e6,step=.01)
        self.compute_feret=self.check('Calculate minimum Feret diameter',True)
        self.exclude_border=self.check('Exclude fibers touching the image border',False)
        self.area_min=self.control(widgets.BoundedFloatText,'Minimum CSA (µm²)',0.,min=0,max=1e12,step=50.)
        self.area_max=self.control(widgets.BoundedFloatText,'Maximum CSA (0 = off)',0.,min=0,max=1e12,step=50.)
        self.csa_max=self.control(widgets.BoundedFloatText,'CSA distribution max',7000.,min=.001,max=1e9,step=250.)
        self.csa_step=self.control(widgets.BoundedFloatText,'CSA bin width (µm²)',250.,min=.001,max=1e9,step=50.)
        self.feret_max=self.control(widgets.BoundedFloatText,'Feret distribution max',100.,min=.001,max=1e6,step=2.)
        self.feret_step=self.control(widgets.BoundedFloatText,'Feret bin width (µm)',2.,min=.001,max=1e6,step=1.)
        measurements=card('05','Set fiber measurements','Calibration refers to the reference mask resolution. Changing measurement filters or units does not rerun automatic typing.',[
            self.calibration,self.pixel_size,self.compute_feret,self.exclude_border,self.area_min,self.area_max,
            self.csa_max,self.csa_step,self.feret_max,self.feret_step,
            field_label('Measurement conventions','Minimum Feret uses the convex hull of pixel centers, as in the source notebook. Distribution maxima include an overflow bin. Filters affect reported statistics; the detailed table retains every detected fiber.')])

        self.export_source=self.control(widgets.Dropdown,'Classification source','reviewed',options=[('Reviewed images only','reviewed'),('Initial calls (manual markers = 0)','automatic')])
        self.export_button=self.button('Export fiber measurements','download','primary',lambda:self.start('export'))
        self.export_message=widgets.HTML()
        exports=card('06','Export results','Export CSA and minimum Feret diameter for all fibers, each positive marker and marker combinations. Hybrids can contribute to more than one marker summary.',[
            self.export_source,field_label('Final typing','Use Reviewed images only to include manual calls and corrected automatic calls. Initial calls keep Manual markers at 0. The Inferred category is recalculated from the selected source at every export.'),self.export_button,self.export_message,
            widgets.HTML(value='<div class="mc-output-map"><strong>Automatic/</strong> original per-image marker calls<br>'
                '<strong>Review/</strong> editable mapping workbooks and review status<br>'
                '<strong>Reference_masks/</strong> fixed fiber labels · <strong>Prepared_channels/</strong> aligned normalized fluorescence<br>'
                '<strong>Signal_masks/</strong> Cellpose predictions by marker<br>'
                '<strong>Exports/automatic/</strong> or <strong>Exports/reviewed/</strong> Excel and CSV results<br>'
                '<strong>Log/</strong> settings, checkpoints, errors and backups</div>')])
        footer=widgets.HTML(value='<div class="mc-footer">MyoClass · local processing · '
            f'Cellpose: {html.escape(installed_version("cellpose"))} · Napari: {html.escape(installed_version("napari"))}<br>')
        self.widget=widgets.VBox([self.header,files,preparation,automatic,review,measurements,exports,footer],layout=widgets.Layout(width='100%'))
        self.widget.add_class('myoclass-app')
        self.mask_mode.observe(self.refresh_preparation,names='value')
        self.scale_mode.observe(self.refresh_preparation,names='value')
        self.refresh_preparation()

    def chooser(self,directories=False,path=None,filename='',select_default=False):
        chooser=FileChooser(str(path or Path.cwd()),filename=filename,show_only_dirs=directories,rows=5,select_default=select_default)
        chooser.layout=widgets.Layout(width='100%'); chooser.add_class('mc-chooser')
        return chooser

    def control(self,cls,description,value,**kwargs):
        widget=cls(description=description,value=value,style=self.style,layout=widgets.Layout(width='620px',max_width='100%'),**kwargs)
        self.controls.append(widget)
        return widget

    def check(self,description,value):
        widget=widgets.Checkbox(description=description,value=value,indent=False,layout=widgets.Layout(width='100%'))
        self.controls.append(widget)
        return widget

    def button(self,description,icon,style,callback):
        widget=widgets.Button(description=description,icon=icon,layout=widgets.Layout(width='230px',height='44px'))
        widget.add_class(f'mc-{style}-button'); widget.on_click(lambda _:callback())
        self.action_buttons.append(widget)
        return widget

    def add_marker(self,index=None,name=None,enabled=True,mode='automatic'):
        if self.busy or self.napari_running(): return
        index=index if index is not None else max([r['index'].value for r in self.marker_rows],default=-1)+1
        checkbox=widgets.Checkbox(value=enabled,description='Use',indent=False,layout=widgets.Layout(width='85px'))
        number=widgets.BoundedIntText(value=index,min=0,max=99,description='Channel',style={'description_width':'65px'},layout=widgets.Layout(width='155px',max_width='100%'))
        text=widgets.Text(value=name or f'Marker {index}',description='Marker / type',style={'description_width':'100px'},layout=widgets.Layout(width='480px',max_width='100%'))
        method=widgets.Dropdown(value=mode,options=[('Automatic (Cellpose)','automatic'),('Manual (Napari)','manual'),('Inferred from other negatives','inferred')],description='Mode',style={'description_width':'45px'},layout=widgets.Layout(width='310px',max_width='100%'))
        row=widgets.VBox([widgets.HBox([checkbox,number,text],layout=widgets.Layout(width='100%',flex_flow='row wrap',gap='8px')),method],layout=widgets.Layout(width='100%',gap='4px'))
        self.marker_rows.append({'enabled':checkbox,'index':number,'name':text,'mode':method,'widget':row})
        self.marker_box.children=tuple(r['widget'] for r in self.marker_rows)

    @staticmethod
    def status_html(state,text):
        return f'<div class="mc-status mc-{state}"><span class="mc-status-dot"></span>{html.escape(text)}</div>'

    def napari_running(self):
        return self.napari_process is not None and self.napari_process.poll() is None

    def get_settings(self):
        settings={key:getattr(self,key).value for key in (
            'recursive','axes','all_scenes','time','z','align_to_mask','mask_mode','edge_only',
            'scale_mode','mask_strength','fixed_scale','gaussian_sigma','normalize_nonzero','diameter','device',
            'flow_threshold','cellprob_threshold','min_size','batch_size','cellpose_normalize','overlap','force',
            'calibration','pixel_size','compute_feret','exclude_border','area_min','area_max','csa_max','csa_step',
            'feret_max','feret_step','export_source')}
        settings.update({'input_dir':self.input_chooser.selected or '', 'mask_dir':self.mask_chooser.selected or '',
                         'model':self.model_chooser.selected or '', 'output_dir':self.output_path.value.strip(),
                         'preview_size':1400,'channels':[{'index':r['index'].value,'name':r['name'].value.strip(),'mode':r['mode'].value} for r in self.marker_rows if r['enabled'].value]})
        return settings

    def show_diameter_guide(self):
        if self.busy or self.napari_running(): return
        try:
            settings=validate_auto(self.get_settings(),require_model=False)
            tasks,issues=inspect_pairs(settings)
            if not tasks: raise ValueError(issues[0]["Error"] if issues else "No matched image/mask pair.")
            task=tasks[0]
            ref=validate_mask(np.squeeze(tiff.imread(task["mask_path"])))
            planes=prepare_channels(read_channels(task["image_path"],task["scene"],settings),ref,settings)
            modes={f"ch{c['index']}":channel_mode(c) for c in settings["channels"]}
            target=next((name for name in planes if modes.get(name)=="automatic"),next(iter(planes)))
            arr=np.asarray(planes[target],dtype=np.float32)
            h,w=arr.shape;side=int(self.diameter_guide_crop.value)
            cy,cx=h//2,w//2;y0=max(0,min(h-side,cy-side//2));x0=max(0,min(w-side,cx-side//2))
            crop=arr[y0:y0+side,x0:x0+side]
            valid=crop[np.isfinite(crop)]
            lo,hi=np.percentile(valid,[1,99.5]) if valid.size else (0,1)
            gray=np.uint8(np.clip((crop-lo)/max(float(hi-lo),1.e-6),0,1)*255)
            from PIL import ImageDraw
            image=Image.fromarray(gray).convert("RGB")
            x,y=image.size[0]//2,image.size[1]//2;radius=float(self.diameter.value)/2
            ImageDraw.Draw(image).ellipse((x-radius,y-radius,x+radius,y+radius),outline=(255,192,45),width=3)
            image.thumbnail((520,520));buff=io.BytesIO();image.save(buff,'PNG')
            self.diameter_guide_image.value=buff.getvalue()
            self.diameter_guide_message.value=f'<div class="mc-note">{html.escape(task["filename"])} · {target} · diameter {self.diameter.value:g} pixels in prepared-channel coordinates. Choose a representative marker-positive signal object.</div>'
        except Exception as exc:
            self.diameter_guide_message.value=f'<div class="mc-note">{html.escape(str(exc))}</div>'

    def on_event(self,event):
        if event['kind']=='log':
            self.messages.append(event['text']); self.messages=self.messages[-12:]
            self.log.value='<div class="mc-log">'+''.join(f'<div>{html.escape(t)}</div>' for t in self.messages)+'</div>'
        elif event['kind']=='file':
            self.file_states[event['file']]=event['state']
            rows=list(self.file_states.items())[-150:]
            self.file_status.value='<div class="mc-files">'+''.join(f'<div class="mc-file"><span>{html.escape(n)}</span><b>{html.escape(st)}</b></div>' for n,st in reversed(rows))+'</div>'
        elif event['kind']=='progress':
            self.progress.value=event['current']/max(1,event['total'])
            self.progress_label.value=f'<div class="mc-note">{html.escape(event["text"])}</div>'
        elif event['kind']=='preview':
            self.preview_image.value=event['png']; self.preview_image.layout.display='block'
            self.preview_caption.value=f'<div class="mc-note">{html.escape(event["text"])}</div>'

    def lock(self,locked):
        for widget in self.controls: widget.disabled=locked
        for row in self.marker_rows:
            for key in ('enabled','index','name','mode'): row[key].disabled=locked
        for button in self.action_buttons: button.disabled=locked
        self.review_image.disabled=locked
        if locked: self.widget.add_class('mc-busy')
        else:
            self.widget.remove_class('mc-busy')
            self.refresh_preparation()

    def refresh_preparation(self,*_):
        if self.busy or self.napari_running(): return
        subtract=self.mask_mode.value=='subtract'
        for widget in (self.edge_only,self.scale_mode,self.mask_strength,self.gaussian_sigma):
            widget.disabled=not subtract
        self.fixed_scale.disabled=not (subtract and self.scale_mode.value=='fixed')

    def refresh_project(self):
        try:
            root=project_paths(self.get_settings())
            entries=available_entries(root)
            current=self.review_image.value
            options=[(f"{e['filename']} · scene {e['scene']+1} · {get_review_state(e)}",e['image_id']) for e in entries]
            self.review_image.options=options or [('No current automatic mappings found','')]
            if current in [value for _,value in options]: self.review_image.value=current
            reviewed=sum(get_review_state(e)=='Reviewed' for e in entries)
            self.review_summary.value=f'<div class="mc-note"><strong>{len(entries)}</strong> classified image(s)/scene(s) · <strong>{reviewed}</strong> reviewed · <strong>{len(entries)-reviewed}</strong> draft or stale</div>'
        except Exception as exc:
            self.review_message.value=f'<div class="mc-note">{html.escape(str(exc))}</div>'

    def reset_selected(self):
        if self.busy or self.napari_running(): return
        try:
            root=project_paths(self.get_settings())
            entry=next(e for e in available_entries(root) if e['image_id']==self.review_image.value)
            initialize_review(entry,reset=True)
            save_review_session(root)
            self.review_message.value='<div class="mc-note">Selected review reset from automatic calls. Previous edits were backed up in Review/Backups.</div>'
            self.refresh_project()
        except Exception as exc:
            self.review_message.value=f'<div class="mc-note">{html.escape(str(exc) or "Select a valid classified image.")}</div>'

    def load_existing(self,settings):
        root=project_paths(settings)
        project=read_project(root)
        saved=project.get('settings')
        if not saved: raise ValueError('No existing MyoClass project found in this output folder.')
        self.output_path.value=str(root)
        for attribute,key,directories in [('input_chooser','input_dir',True),('mask_chooser','mask_dir',True),('model_chooser','model',False)]:
            path=Path(saved.get(key,''))
            if path.exists():
                chooser=self.chooser(directories,path=path if directories else path.parent,filename='' if directories else path.name,select_default=True)
                setattr(self,attribute,chooser)
                box=getattr(self,{'input_chooser':'input_box','mask_chooser':'mask_box','model_chooser':'model_box'}[attribute])
                box.children=(box.children[0],chooser)
        self.marker_rows=[]
        for spec in normalize_channel_specs(saved['channels'],saved.get('infer_last',False)):
            self.add_marker(spec['index'],spec['name'],mode=spec['mode'])
        for key,value in saved.items():
            widget=getattr(self,key,None)
            if key not in ('channels','output_dir') and hasattr(widget,'value'):
                try: widget.value=value
                except (TypeError,ValueError): pass
        self.force.value=False
        self.refresh_project()
        self.status.value=self.status_html('complete','Existing project loaded. Review files have been kept.')

    def start(self,action):
        if self.busy or self.napari_running(): return
        try:
            settings=self.get_settings()
            if action=='load':
                self.load_existing(settings); return
            if action in ('run','test'): settings=validate_auto(settings)
            elif action in ('preview','pairs'): settings=validate_auto(settings,require_model=False)
            elif action=='export': validate_export(settings)
            loop=asyncio.get_running_loop()
        except Exception as exc:
            self.status.value=self.status_html('error',str(exc)); return
        self.busy=True; self.lock(True); self.stop_event.clear(); self.messages=[]; self.log.value=''
        self.progress.value=0; self.stop_button.disabled=action not in ('run','test')
        self.status.value=self.status_html('running',{'export':'Exporting measurements…','preview':'Preparing preview…','pairs':'Checking image/mask pairs…'}.get(action,'Automatic typing running…'))
        self.task=loop.create_task(self.execute(action,settings,loop))

    async def execute(self,action,settings,loop):
        try:
            def emit(event): loop.call_soon_threadsafe(self.on_event,event)
            if action=='preview':
                self.on_event(await loop.run_in_executor(None,lambda:selected_preview(settings)))
                self.status.value=self.status_html('complete','Prepared preview ready. Check signal and mask alignment.')
            elif action=='pairs':
                pairs,issues=await loop.run_in_executor(None,lambda:inspect_pairs(settings))
                for pair in pairs[:8]: self.on_event({'kind':'log','text':f"{pair['filename']} · scene {pair['scene']+1} → {Path(pair['mask_path']).name}"})
                for issue in issues[:8]: self.on_event({'kind':'log','text':f"{issue['Filename']}: {issue['Error']}"})
                self.status.value=self.status_html('error' if issues else 'complete',f'{len(pairs)} matched image(s)/scene(s) · {len(issues)} pairing issue(s)')
            elif action=='export':
                result=await loop.run_in_executor(None,lambda:export_measurements(settings,emit))
                self.export_message.value=f'<div class="mc-note"><strong>{result["images"]} image(s), {result["fibers"]} included fibers</strong><br>{html.escape(result["workbook"])}<br>{result["issues"]} skipped/pending issue(s); see Export_issues.</div>'
                self.status.value=self.status_html('complete',f'Export finished · {result["issues"]} issue(s) listed')
            else:
                result=await loop.run_in_executor(None,lambda:run_automatic(settings,emit,self.stop_event,action=='test'))
                self.output_path.value=result['output_dir']
                self.status.value=self.status_html('error' if result['errors'] else 'complete',
                    f"{'Stopped' if result['stopped'] else 'Typing finished'} · {result['successful']} successful · {result['errors']} error(s) · {result['reused']} reused prediction(s)")
            self.progress.value=1
        except Exception as exc:
            self.status.value=self.status_html('error',str(exc))
            self.on_event({'kind':'log','text':f'ERROR: {exc}'})
        finally:
            self.busy=False; self.lock(False); self.stop_button.disabled=True
            self.refresh_project()

    def stop(self,_):
        if self.busy:
            self.stop_event.set(); self.stop_button.disabled=True
            self.status.value=self.status_html('running','Stop requested. Finishing the current image and saving checkpoints…')

    def open_editor(self):
        if self.busy or self.napari_running(): return
        try:
            root=project_paths(self.get_settings())
            loop=asyncio.get_running_loop()
            image_id=self.review_image.value
            executable=self.napari_python.value
            self.busy=True
            self.lock(True)
            self.review_message.value='<div class="mc-note">Checking the selected Python environment, Qt binding and review workbook dependencies…</div>'
            self.napari_watch_task=loop.create_task(self.start_editor(root,image_id,executable,loop))
        except Exception as exc:
            self.busy=False
            self.lock(False)
            self.review_message.value=f'<div class="mc-note">{html.escape(str(exc))}</div>'

    async def start_editor(self,root,image_id,executable,loop):
        try:
            process,path=await loop.run_in_executor(None,lambda:launch_napari_project(root,image_id,REVIEW_SCRIPT_SOURCE,executable))
            self.napari_process=process
            self.busy=False
            self.review_message.value='<div class="mc-note">Napari is running in a desktop window. Save drafts or mark images reviewed, then close Napari to enable final export.</div>'
            await self.watch_editor(path)
        except Exception as exc:
            self.busy=False
            self.lock(False)
            self.review_message.value=f'<div class="mc-note"><pre>{html.escape(str(exc))}</pre></div>'

    async def watch_editor(self,path):
        while self.napari_running(): await asyncio.sleep(1)
        code=self.napari_process.returncode
        self.lock(False); self.refresh_project()
        if code:
            lines=Path(path).read_text(encoding='utf-8',errors='replace').splitlines()
            details='\n'.join(lines[:4]+['…']+lines[-18:])
            self.review_message.value=f'<div class="mc-note">Napari exited with code {code}. Startup diagnostics and traceback:<br><pre>{html.escape(details)}</pre></div>'
        else:
            self.review_message.value='<div class="mc-note">Napari closed. Review status refreshed; you can now export reviewed results.</div>'


myoclass_app=MyoClassApp()
display(myoclass_app.widget)
